# v28 — v25 80% + v26 10% + v27 10%의 그룹 5-fold 확인

v25의 사용자 확인 Public은 **0.390417094**입니다. 목표 0.50은 실제 제출 점수 목표이며 로컬 0.50 달성과 구분합니다. 기존 3-fold OOF의 사전 고정 조합 탐색 결과 0.500519는 이 확인 실험과 별도로 기록합니다.

비교 환자를 고정하기 위해 기존 개발 4,960명을 seed28028 그룹 5-fold로 재분할합니다. 각 모델은 개발 환자의 80%로 학습하며 보류 1,241명은 평가하지 않습니다. 이는 독립 외부 환자 검증이 아닌 같은 개발 환자의 재분할입니다. 모든 구성 모델을 해당 fold에서 새로 fit합니다. 형제 라벨 교정과 COAD prior는 사용하지 않습니다.

아래에는 구성 모델의 모든 소스·선별·학습·검증·저장·최종 추론을 포함합니다. 별도 프로젝트 Python 파일이나 이전 노트북은 필요하지 않습니다. 기존 artifacts가 없으면 구성 모델을 재학습하고 GF 표는 고정 공식 리비전에서 취득합니다. 처음 취득에는 네트워크가 필요합니다. CPU/RAM을 고려해 독립 worker를 최대 2개 실행합니다. test/sample은 마지막 고정 모델 추론에서만 사용합니다.


In [1]:
COMPONENT_SOURCES28={'25': ["import os,sys,re,json,pickle,cloudpickle,hashlib,platform,time,copy,warnings\nfrom pathlib import Path\nfrom collections import Counter\nimport numpy as np\nimport pandas as pd\nfrom scipy import sparse\nfrom scipy.special import softmax\nfrom sklearn.base import BaseEstimator,TransformerMixin,clone\nfrom sklearn.feature_extraction import DictVectorizer\nfrom sklearn.feature_selection import chi2,f_classif\nfrom sklearn.preprocessing import StandardScaler,MaxAbsScaler\nfrom sklearn.multiclass import OneVsRestClassifier\nfrom sklearn.model_selection import StratifiedGroupKFold\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.ensemble import ExtraTreesClassifier\nfrom sklearn.metrics import f1_score,accuracy_score,classification_report,confusion_matrix\nfrom lightgbm import LGBMClassifier\nfrom threadpoolctl import threadpool_limits\nfrom IPython.display import display,FileLink\nthreadpool_limits(2)\nSEED=42\nforbidden_attempts23=[]\ndef forbid_inference23(event,args):\n if event=='open' and args and isinstance(args[0],(str,bytes,os.PathLike)) and Path(os.fsdecode(args[0])).name in {'test.csv','sample_submission.csv'}:\n  forbidden_attempts23.append(str(args[0]));raise RuntimeError('Inference data access is forbidden in v23')\nsys.addaudithook(forbid_inference23)\nroots=[Path.cwd(),*Path.cwd().parents]\nREPO=next((p for p in roots if (p/'data/train.csv').is_file()),None)\nif REPO is None:raise FileNotFoundError('data/train.csv가 필요합니다.')\nDATA=REPO/'data';WORK=REPO/'models'/'혜진' if (REPO/'models'/'혜진').is_dir() else REPO\nOUT=WORK/'artifacts'/'v28'/'definitions_v25';OUT.mkdir(parents=True,exist_ok=True)\n# Inference files and submission directory are intentionally not used.\nos.environ['MPLCONFIGDIR']=str(OUT/'.mplconfig')\ntrain=pd.read_csv(DATA/'train.csv')\ngenes=[c for c in train if c not in ['ID','SUBCLASS']]\nassert train.ID.is_unique\ny=train.SUBCLASS.to_numpy();labels=np.sort(np.unique(y));yi=np.searchsorted(labels,y);NC=len(labels)\n\ndef normalize_token25(token):\n return re.sub(r'^(p\\.)?([ACDEFGHIKLMNPQRSTVWY])([1-9][0-9]*)X$',lambda m:(m[1] or '')+m[2]+m[3]+'*',token)\ndef canonical_frame25(frame):\n def one(v):\n  if not isinstance(v,str) or 'X' not in v:return v\n  return ' '.join(normalize_token25(t) for t in v.split())\n return frame.map(one)\ndef legacy_frame25(frame,mixed=False):\n def one(v):\n  if not isinstance(v,str) or '*' not in v:return v\n  def conv(m):\n   token=m[0]\n   if mixed and int(hashlib.sha256(token.encode()).hexdigest()[:8],16)%2:return token\n   return token[:-1]+'X'\n  return re.sub(r'(?<!\\S)(?:p\\.)?[ACDEFGHIKLMNPQRSTVWY][1-9][0-9]*\\*(?!\\S)',conv,v)\n return frame.map(one)\n\nX25_original=train[genes].copy();X25_mixed=legacy_frame25(X25_original,mixed=True);X=canonical_frame25(X25_mixed);assert X.equals(X25_original);groups=pd.util.hash_pandas_object(X,index=False).to_numpy()\ndev,hold=next(StratifiedGroupKFold(5,shuffle=True,random_state=SEED).split(X,y,groups))\nfolds=list(StratifiedGroupKFold(3,shuffle=True,random_state=SEED+1).split(X.iloc[dev],y[dev],groups[dev]))\nassert not set(groups[dev]).intersection(groups[hold])\nfor a,b in folds:assert not set(groups[dev[a]]).intersection(groups[dev[b]])\npd.DataFrame({'ID':train.ID,'split':np.where(np.isin(np.arange(len(train)),hold),'holdout','development'),'group':groups.astype(str)}).to_csv(OUT/'split_manifest.csv',index=False)\nprint('TRAIN ONLY',train.shape,'Dev/holdout',len(dev),len(hold),'OUT',OUT,flush=True)\n\nfrom sklearn.neural_network import MLPClassifier\nfrom sklearn.feature_extraction.text import TfidfTransformer\nfrom sklearn.utils.class_weight import compute_sample_weight\nfrom xgboost import XGBClassifier\n", "GROUPS = {\n 'p53_cellcycle': 'TP53 RB1 CDKN2A CDKN2B CDK4 CDK6 CCND1 ATM CHEK2'.split(),\n 'rtk_ras_mapk': 'EGFR ERBB2 ERBB3 KRAS NRAS HRAS BRAF NF1 MAP2K1 MAP2K2 KIT PDGFRA FGFR2 FGFR3'.split(),\n 'pi3k_mtor': 'PIK3CA PIK3R1 PTEN AKT1 MTOR TSC1 TSC2 STK11'.split(),\n 'dna_repair': 'BRCA1 BRCA2 PALB2 ATM ATR CHEK2 MLH1 MSH2 MSH6 PMS2 POLE POLD1'.split(),\n 'chromatin': 'ARID1A ARID1B ARID2 PBRM1 SMARCA4 BAP1 SETD2 KMT2A KMT2C KMT2D ATRX'.split(),\n 'wnt': 'APC CTNNB1 AXIN1 AXIN2 RNF43'.split(),\n 'metabolism': 'IDH1 IDH2 SDHB SDHC SDHD FH VHL'.split(),\n}\n\ndef kind(token):\n if 'fs' in token.lower(): return 'frameshift'\n if '*' in token: return 'stop'\n if 'del' in token.lower(): return 'deletion'\n if 'ins' in token.lower(): return 'insertion'\n m = re.fullmatch(r'([A-Z])(\\d+)([A-Z])', token)\n if m: return 'synonymous' if m[1] == m[3] else 'substitution'\n return 'other'\n\nclass MutationFeatures(BaseEstimator, TransformerMixin):\n def __init__(self, mode='full', min_df=3):\n  self.mode = mode\n  self.min_df = min_df\n\n def dictionaries(self, X):\n  genes = list(X.columns)\n  for row in X.itertuples(index=False, name=None):\n   d = {}; counts = {}; burden = 0; events = 0; positions = []\n   changed = set()\n   for gene, value in zip(genes, row):\n    if value is None or str(value) in ('nan', '', '<NA>'):\n     d['missing:' + gene] = 1; continue\n    if value == 'WT': continue\n    changed.add(gene); burden += 1; d['gene:' + gene] = 1\n    if self.mode == 'binary': continue\n    tokens = str(value).split()\n    for raw_token in tokens:\n     token = re.sub(r'^([A-Z]\\d+)X$', r'\\1*', raw_token)\n     events += 1; typ = kind(token); counts[typ] = counts.get(typ, 0) + 1\n     d['variant:' + gene + ':' + token] = 1\n     d['type:' + gene + ':' + typ] = 1\n     m = re.search(r'\\d+', token)\n     if m:\n      position = int(m[0]); positions.append(position)\n      d['position:' + gene + ':' + str(position)] = 1\n   if self.mode == 'full':\n    d['numeric:log_mutated_genes'] = np.log1p(burden)\n    d['numeric:log_events'] = np.log1p(events)\n    d['numeric:multi_event_excess'] = np.log1p(max(0, events-burden))\n    for typ in ['frameshift','stop','deletion','insertion','synonymous','substitution','other']:\n     d['numeric:log_' + typ] = np.log1p(counts.get(typ,0))\n     d['numeric:fraction_' + typ] = counts.get(typ,0)/max(events,1)\n    d['numeric:log_mean_position'] = np.log1p(np.mean(positions)) if positions else 0.\n    for name, members in GROUPS.items():\n     d['module:' + name] = np.log1p(len(changed.intersection(members)))\n   yield d\n\n def fit(self, X, y=None):\n  self.genes_ = list(X.columns)\n  self.vectorizer_ = DictVectorizer(dtype=np.float32)\n  matrix = self.vectorizer_.fit_transform(self.dictionaries(X))\n  names = self.vectorizer_.get_feature_names_out()\n  self.keep_ = (np.asarray((matrix != 0).sum(axis=0)).ravel() >= self.min_df) | np.array([s.startswith(('numeric:', 'module:')) for s in names])\n  self.feature_names_ = names[self.keep_]\n  return self\n\n def transform(self, X):\n  missing = set(self.genes_) - set(X.columns)\n  if missing: raise ValueError('Missing gene columns: ' + str(sorted(missing)[:10]))\n  return sparse.csr_matrix(self.vectorizer_.transform(self.dictionaries(X[self.genes_]))[:, self.keep_])\n\n def get_feature_names_out(self, input_features=None): return self.feature_names_\ndef canon(t):\n t=t.strip().removeprefix('p.');t=re.sub(r'(?<=\\d)X$','*',t)\n return re.sub(r'^([A-Z]\\d+)(?:[A-Z*]+)?fs.*$',r'\\1fs',t)\ndef aa_change(t):\n m=re.fullmatch(r'([A-Z])\\d+([A-Z*])',t)\n if m:return m[1]+'>'+m[2]\n m=re.match(r'([A-Z])\\d+',t)\n if m:\n  for suffix in ['fs','del','ins']:\n   if suffix in t:return m[1]+'>'+suffix\n return None\n\ndef parse_events(frame):\n names=list(frame.columns);result=[]\n for row in frame.itertuples(index=False,name=None):\n  result.append([(g,tuple(sorted(set(canon(t) for t in v.split())))) for g,v in zip(names,row) if isinstance(v,str) and v!='WT'])\n return result\n\nclass EvidenceEncoder:\n def __init__(self,unknown=1.,silent=1.,support=2):self.unknown=unknown;self.silent=silent;self.support=support\n def fit(self,events):\n  self.token_df={};self.change_df={}\n  if self.unknown<1:\n   tokens=Counter();changes=Counter()\n   for row in events:\n    for g,ts in row:\n     tokens.update((g,t) for t in ts)\n     changes.update(set((g,a) for t in ts if (a:=aa_change(t)) is not None))\n   self.token_df={k:v for k,v in tokens.items() if v>=self.support}\n   self.change_df={k:v for k,v in changes.items() if v>=self.support}\n  return self\n def transform(self,events,training=False,exclude_events=None):\n  result=[]\n  for row_number,row in enumerate(events):\n   own=exclude_events[row_number] if exclude_events is not None else (row if training else [])\n   own_tokens={(gg,tt) for gg,tts in own for tt in tts}\n   own_changes={(gg,ac) for gg,tts in own for tt in tts if (ac:=aa_change(tt)) is not None}\n   d={};gene_weights={};type_sums=Counter();non_syn=0.;trunc=0.;multikind=0\n   for g,ts in row:\n    types={};gene=0.;known_gene=False\n    for t in ts:\n     typ=kind(t);a=aa_change(t)\n     known=self.token_df.get((g,t),0)-int((g,t) in own_tokens)>=self.support\n     known_aa=a is not None and self.change_df.get((g,a),0)-int((g,a) in own_changes)>=self.support\n     if typ=='synonymous':w=self.silent\n     elif known or typ in ['stop','frameshift','deletion','insertion']:w=1.\n     elif known_aa:w=max(self.unknown,.75)\n     else:w=self.unknown\n     if w<=0:continue\n     gene=max(gene,w);types[typ]=max(types.get(typ,0),w)\n     d['variant:'+g+':'+t]=max(d.get('variant:'+g+':'+t,0),w)\n     m=re.search(r'\\d+',t)\n     if m:d['position:'+g+':'+m[0]]=max(d.get('position:'+g+':'+m[0],0),w)\n     if a:d['aa:'+g+':'+a]=max(d.get('aa:'+g+':'+a,0),w)\n     if known and typ!='synonymous':known_gene=True\n    if gene<=0:continue\n    d['gene:'+g]=gene;gene_weights[g]=gene\n    for typ,w in types.items():d['type:'+g+':'+typ]=w;type_sums[typ]+=w\n    if known_gene:d['supported:'+g]=1.\n    non_syn+=max([w for k,w in types.items() if k!='synonymous'] or [0.])\n    trunc+=max([w for k,w in types.items() if k in ['stop','frameshift','deletion','insertion']] or [0.])\n    multikind+=len(types)>1\n   n=len(row)\n   d['numeric:log_raw_genes']=np.log1p(n)\n   d['numeric:log_effective_genes']=np.log1p(sum(gene_weights.values()))\n   d['numeric:log_nonsyn_genes']=np.log1p(non_syn)\n   d['numeric:log_trunc_genes']=np.log1p(trunc)\n   d['numeric:log_multikind_genes']=np.log1p(multikind)\n   for typ in ['synonymous','substitution','stop','frameshift','deletion','insertion','other']:\n    d['numeric:gene_fraction_'+typ]=type_sums[typ]/max(n,1)\n   for name,gs in GROUPS.items():d['module:'+name]=np.log1p(sum(gene_weights.get(g,0) for g in gs))\n   result.append(d)\n  return result\n\nevents=parse_events(X)\nlegacy_records=list(MutationFeatures(mode='full').dictionaries(X))\nassert canon('R213X')=='R213*' and canon('K16Rfs')=='K16fs'\nassert aa_change('R132H')==aa_change('R140H')=='R>H'\nprint('Parsed rows:',len(events),'Normalization examples passed')\n", "def relevance_scores(a,target,method):\n base=method.removesuffix('_full')\n if base=='embedded':\n  scaled=csr32(MaxAbsScaler().fit_transform(a))\n  fitted=OneVsRestClassifier(LogisticRegression(solver='liblinear',l1_ratio=1,C=.2,class_weight='balanced',max_iter=2000),n_jobs=1).fit(scaled,target)\n  return np.max(np.abs(np.vstack([e.coef_[0] for e in fitted.estimators_])),axis=0)\n return (chi2(a,target) if base=='chi2' else f_classif(a,target))[0]\n\ndef csr32(a):\n b=sparse.csr_matrix(a,dtype=np.float32)\n if b.nnz>=np.iinfo(np.int32).max:raise ValueError('Sparse matrix too large for tree index type')\n b.indices=b.indices.astype(np.int32,copy=False);b.indptr=b.indptr.astype(np.int32,copy=False)\n return b\n\nclass Space:\n def fit(self,rs):\n  self.vec=DictVectorizer(dtype=np.float32);a=self.vec.fit_transform(rs)\n  names=self.vec.get_feature_names_out();df=np.asarray((a!=0).sum(axis=0)).ravel()\n  keep=(df>=3)|np.array([n.startswith(('numeric:','module:')) for n in names])\n  self.raw=np.flatnonzero(keep);self.names=names[keep]\n  return csr32(a[:,self.raw])\n def transform(self,rs):return csr32(self.vec.transform(rs)[:,self.raw])\n def compact_vec(self,indices):\n  v=copy.deepcopy(self.vec);mask=np.zeros(len(v.feature_names_),bool);mask[self.raw[np.sort(indices)]]=True\n  v.restrict(mask);return v\n\nclass Selector:\n def __init__(self,a,target,names):\n  self.a=a;self.y=target;self.names=names;self.cache={};self.corr={};self.scores={};self.audit=[];self.examples=[]\n def order(self,method,threshold):\n  key=(method,threshold)\n  if key in self.cache:return self.cache[key]\n  if method not in self.scores:\n   with warnings.catch_warnings():\n    warnings.simplefilter('ignore',RuntimeWarning);score=relevance_scores(self.a,self.y,method)\n   score=np.nan_to_num(score,nan=0,posinf=np.finfo(np.float32).max,neginf=0)\n   if not method.endswith('_full'):score[np.array([n.startswith('numeric:') for n in self.names])]=0\n   self.scores[method]=score\n  score=self.scores[method]\n  if method not in self.corr:\n   pool=np.argsort(-score,kind='stable')[:min(4096,int((score>0).sum()))]\n   z=self.a[:,pool];mean=np.asarray(z.mean(axis=0)).ravel();var=np.asarray(z.multiply(z).mean(axis=0)).ravel()-mean**2\n   sd=np.sqrt(np.maximum(var,1e-12));corr=(z.T@z).toarray()/z.shape[0]-np.outer(mean,mean)\n   corr/=np.outer(sd,sd);np.clip(corr,-1,1,out=corr)\n   self.corr[method]=(pool,corr)\n  pool,corr=self.corr[method];chosen=[]\n  for i in range(len(pool)):\n   if chosen:\n    correlations=np.abs(corr[i,chosen]);j=int(np.argmax(correlations))\n    if correlations[j]>=threshold:\n     if len(self.examples)<100:self.examples.append({'method':method,'threshold':threshold,'kept':self.names[pool[chosen[j]]],'discarded':self.names[pool[i]],'correlation':float(corr[i,chosen[j]])})\n     continue\n   chosen.append(i)\n   if len(chosen)>=2048:break\n  order=pool[chosen];self.cache[key]=order\n  self.audit.append({'method':method,'threshold':threshold,'association_pool':len(pool),'remaining_up_to_2048':len(order)})\n  return order\n def select(self,method,threshold,k):return np.sort(self.order(method,threshold)[:k])\n\nclass CompactPredictor:\n def __init__(self,space,cols,est):\n  self.vec=space.compact_vec(cols);self.names=self.vec.get_feature_names_out();self.est=est\n  self.direct_inputs=len(self.names);self.raw_inputs=len(self.names);self.kind='compact'\n def predict(self,rs):return self.est.predict_proba(csr32(self.vec.transform(rs)))\n\ndef background_frame(frame,training_frame,seed,count):\n rng=np.random.default_rng(seed);out=frame.copy();arr=out.to_numpy(copy=True)\n rate=(training_frame.notna()&training_frame.ne('WT')).mean().to_numpy();prob=rate/rate.sum();idx=np.flatnonzero(prob>0)\n for r in range(len(arr)):\n  n=min(len(idx),int(rng.poisson(count)))\n  for c in rng.choice(len(prob),size=n,replace=False,p=prob):\n   token='A1A' if rng.random()<.5 else 'A1V'\n   arr[r,c]=token if not isinstance(arr[r,c],str) or arr[r,c]=='WT' else arr[r,c]+' '+token\n return pd.DataFrame(arr,columns=frame.columns,index=frame.index)\nclass ColumnPost:\n def __init__(self,mode='none'):self.mode=mode\n def fit(self,a,names):\n  self.raw_names=np.array(names);self.names=np.array(names)\n  if self.mode=='maxabs':self.scaler=MaxAbsScaler().fit(a)\n  if self.mode=='block':\n   self.binary=np.flatnonzero([not n.startswith(('numeric:','module:')) for n in names])\n   self.summary=np.flatnonzero([n.startswith(('numeric:','module:')) for n in names])\n   self.tf=TfidfTransformer(norm='l2',use_idf=True).fit(a[:,self.binary])\n   self.scaler=MaxAbsScaler().fit(a[:,self.summary])\n   self.names=np.concatenate([self.raw_names[self.binary],self.raw_names[self.summary]])\n  return self\n def transform(self,a):\n  if self.mode=='maxabs':return csr32(self.scaler.transform(a))\n  if self.mode=='block':return csr32(sparse.hstack([self.tf.transform(a[:,self.binary]),self.scaler.transform(a[:,self.summary])],format='csr'))\n  return csr32(a)\n\nclass Member:\n def __init__(self,name,view,encoder,space,cols,post,est):\n  self.name=name;self.view=view;self.encoder=encoder;self.vec=space.compact_vec(cols);self.post=post;self.est=est\n  self.raw_names=self.vec.get_feature_names_out();self.names=post.names\n def predict_rows(self,rs):return self.est.predict_proba(self.post.transform(csr32(self.vec.transform(rs))))\n\nVIEWS={'raw':(1.,1.),'syn_down':(1.,.25),'gated':(.25,1.),'gated_syn_down':(.25,.25),'gated_nonsyn':(.25,0.)}\nCONFIGS={}\nfor view in VIEWS:\n for k in [1024,2048]:\n  for c in [.1,.5]:CONFIGS[f'{view}_lr_k{k}_C{c}']=dict(view=view,algo='lr',k=k,C=c,post='none',method='chi2_full')\n  CONFIGS[f'{view}_lgb_k{k}']=dict(view=view,algo='lgb',k=k,post='none',method='f_classif_full')\nfor view in ['raw','gated_syn_down']:\n for depth in [3,5]:CONFIGS[f'{view}_xgb_d{depth}']=dict(view=view,algo='xgb',k=2048,depth=depth,post='none',method='f_classif_full')\n for alpha in [1,10]:CONFIGS[f'{view}_mlp_a{alpha}']=dict(view=view,algo='mlp',k=1024,alpha=alpha,post='maxabs',method='chi2_full')\n for c in [1,4]:CONFIGS[f'{view}_blocklr_C{c}']=dict(view=view,algo='lr',k=2048,C=c,post='block',method='chi2_full')\n CONFIGS[f'{view}_extra']=dict(view=view,algo='extra',k=1024,post='none',method='f_classif_full')\n(OUT/'configurations.json').write_text(json.dumps(CONFIGS,indent=2))\n\ndef estimator(cfg):\n algo=cfg['algo']\n if algo=='lr':return LogisticRegression(C=cfg['C'],class_weight='balanced',max_iter=2000)\n if algo=='lgb':return LGBMClassifier(n_estimators=300,learning_rate=.04,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=SEED)\n if algo=='xgb':return XGBClassifier(n_estimators=300,max_depth=cfg['depth'],learning_rate=.04,min_child_weight=3,reg_lambda=10,subsample=.85,colsample_bytree=.85,max_bin=63,tree_method='hist',objective='multi:softprob',eval_metric='mlogloss',n_jobs=2,random_state=SEED)\n if algo=='mlp':return MLPClassifier(hidden_layer_sizes=(128,),activation='relu',alpha=cfg['alpha'],learning_rate_init=.001,batch_size=128,max_iter=250,tol=1e-4,n_iter_no_change=15,early_stopping=False,random_state=SEED)\n return ExtraTreesClassifier(n_estimators=250,max_depth=20,min_samples_leaf=3,max_features='sqrt',class_weight='balanced',n_jobs=2,random_state=SEED)\n\ndef prepare_view(idx,view):\n ev=[events[i] for i in idx];enc=EvidenceEncoder(*VIEWS[view]).fit(ev)\n rs=enc.transform(ev,training=True);space=Space();a=space.fit(rs);selector=Selector(a,yi[idx],space.names)\n return enc,space,a,selector\n\ndef train_new(name,idx,cache):\n cfg=CONFIGS[name];view=cfg['view']\n if view not in cache:cache[view]=prepare_view(idx,view)\n enc,space,a,selector=cache[view];cols=selector.select(cfg['method'],.98,cfg['k'])\n post=ColumnPost(cfg['post']).fit(a[:,cols],space.names[cols]);tx=post.transform(a[:,cols]);target=yi[idx]\n row_weights=np.ones(len(idx),dtype=float)\n if cfg.get('augment',False):\n  if '_aug_events' not in cache:\n   cache['_aug_events']=parse_events(rare_background(X.iloc[idx],[events[i] for i in idx],SEED+777,count=10))\n  aug_rs=enc.transform(cache['_aug_events'],exclude_events=[events[i] for i in idx])\n  aug_x=post.transform(space.transform(aug_rs)[:,cols]);tx=csr32(sparse.vstack([tx,aug_x],format='csr'))\n  target=np.concatenate([target,target]);row_weights=np.concatenate([row_weights,.5*row_weights])\n est=estimator(cfg)\n if cfg['algo'] in ['xgb','mlp']:est.fit(tx,target,sample_weight=compute_sample_weight('balanced',target)*row_weights)\n elif cfg.get('augment',False):est.fit(tx,target,sample_weight=row_weights)\n else:est.fit(tx,target)\n return Member(name,view,enc,space,cols,post,est)\n\nLEGACY={\n 'legacy_lr':dict(k=2048,method='chi2',algo='lr'),\n 'legacy_lgb':dict(k=2048,method='f_classif_full',algo='lgb'),\n 'legacy_extra':dict(k=1024,method='f_classif_full',algo='extra')}\ndef train_legacy(idx):\n space=Space();a=space.fit([legacy_records[i] for i in idx]);sel=Selector(a,yi[idx],space.names);result=[]\n for name,cfg in LEGACY.items():\n  cols=sel.select(cfg['method'],.95,cfg['k'])\n  if cfg['algo']=='lr':est=LogisticRegression(C=.1,class_weight='balanced',max_iter=2000)\n  elif cfg['algo']=='lgb':est=LGBMClassifier(n_estimators=250,learning_rate=.05,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=SEED)\n  else:est=ExtraTreesClassifier(n_estimators=250,max_depth=20,min_samples_leaf=3,max_features='sqrt',class_weight='balanced',n_jobs=2,random_state=SEED)\n  est.fit(a[:,cols],yi[idx]);post=ColumnPost().fit(a[:,cols],space.names[cols]);result.append(Member(name,'legacy',None,space,cols,post,est))\n return result\n\ndef duplicate_frame(frame):\n arr=frame.to_numpy(copy=True)\n for i,row in enumerate(arr):\n  for j,v in enumerate(row):\n   if isinstance(v,str) and v!='WT':arr[i,j]=v+' '+v\n return pd.DataFrame(arr,columns=frame.columns,index=frame.index)\n\ndef rare_background(frame,tr_events,seed,count=20):\n rng=np.random.default_rng(seed);df=Counter((g,t) for row in tr_events for g,ts in row for t in ts)\n syn=[];sub=[]\n for (g,t),n in df.items():\n  if n>2:continue\n  if kind(t)=='synonymous':syn.append((g,t))\n  elif kind(t)=='substitution':sub.append((g,t))\n arr=frame.to_numpy(copy=True);lookup={g:i for i,g in enumerate(frame.columns)}\n for row in arr:\n  for _ in range(int(rng.poisson(count))):\n   pool=syn if rng.random()<.5 else sub\n   g,t=pool[int(rng.integers(len(pool)))];j=lookup[g]\n   row[j]=t if not isinstance(row[j],str) or row[j]=='WT' else row[j]+' '+t\n return pd.DataFrame(arr,columns=frame.columns,index=frame.index)\n\ndef eval_views(idx,tr_idx,seed):\n clean=X.iloc[idx];noise=rare_background(clean,[events[i] for i in tr_idx],seed);dup=duplicate_frame(clean)\n frames=[clean,noise,dup]\n return frames,[parse_events(f) for f in frames],[list(MutationFeatures(mode='full').dictionaries(f)) for f in frames]\n\ndef predict_members(members,weights,frame):\n ev=None;old=None;cache={};p=None\n for m,w in zip(members,weights):\n  if m.view=='legacy':\n   if old is None:old=list(MutationFeatures(mode='full').dictionaries(frame[genes]))\n   rs=old\n  else:\n   if ev is None:ev=parse_events(frame[genes])\n   if m.view not in cache:cache[m.view]=m.encoder.transform(ev)\n   rs=cache[m.view]\n  q=m.predict_rows(rs);p=q*w if p is None else p+q*w\n return p\nprint('Novel settings:',len(CONFIGS),'Views:',VIEWS)\n\n# CV-guided targeted refinement: regularization, support threshold, and rare-variant augmentation.\nVIEWS['gated_syn_down_s3']=(.25,.25,3)\nCONFIGS.update({'raw_mlp_a0.01': {'view': 'raw', 'algo': 'mlp', 'k': 1024, 'alpha': 0.01, 'post': 'maxabs', 'method': 'chi2_full'}, 'raw_mlp_a0.1': {'view': 'raw', 'algo': 'mlp', 'k': 1024, 'alpha': 0.1, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_mlp_a0.01': {'view': 'gated_syn_down', 'algo': 'mlp', 'k': 1024, 'alpha': 0.01, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_mlp_a0.1': {'view': 'gated_syn_down', 'algo': 'mlp', 'k': 1024, 'alpha': 0.1, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k1024_C0.1': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 1024, 'C': 0.1, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k1024_C0.5': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 1024, 'C': 0.5, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lgb_k1024': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 1024, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_lr_k2048_C0.1': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 2048, 'C': 0.1, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k2048_C0.5': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 2048, 'C': 0.5, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lgb_k2048': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_xgb_d3': {'view': 'gated_syn_down_s3', 'algo': 'xgb', 'k': 2048, 'depth': 3, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_xgb_d5': {'view': 'gated_syn_down_s3', 'algo': 'xgb', 'k': 2048, 'depth': 5, 'post': 'none', 'method': 'f_classif_full'}, 'raw_aug_lgb': {'view': 'raw', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full', 'augment': True}, 'raw_aug_xgb_d5': {'view': 'raw', 'algo': 'xgb', 'k': 2048, 'depth': 5, 'post': 'none', 'method': 'f_classif_full', 'augment': True}, 'gated_s3_aug_lgb': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full', 'augment': True}})\n(OUT/'configurations.json').write_text(json.dumps(CONFIGS,indent=2))\nprint('Total new configurations after refinement:',len(CONFIGS))\n", "# Freeze the previous version; do not repeat its 59-setting search.\nBASE_CONFIGS=copy.deepcopy(CONFIGS)\nBASE_PLAN=[('v4_reference',.25),('gated_s3_aug_lgb',.1875),('raw_aug_xgb_d5',.1875),('raw_lr_k2048_C0.1',.1875),('gated_syn_down_mlp_a1',.1875)]\n\ndef fit_base(idx):\n cache={};ms=[];ws=[]\n for name,w in BASE_PLAN:\n  if name=='v4_reference':ms.extend(train_legacy(idx));ws.extend([w/3]*3)\n  else:ms.append(train_new(name,idx,cache));ws.append(w)\n return ms,np.array(ws)\n", "class NewSelector:\n def __init__(self,a,target,names):self.a=a;self.y=target;self.names=names;self.cache={}\n def select(self,method,k):\n  if (method,k) in self.cache:return self.cache[(method,k)]\n  numeric=np.array([s.startswith(('numeric:','module:')) for s in self.names])\n  with warnings.catch_warnings():\n   warnings.simplefilter('ignore',RuntimeWarning)\n   global_score=np.nan_to_num(f_classif(self.a,self.y)[0],nan=0,posinf=0,neginf=0)\n   if method=='balanced':\n    orders=[]\n    for c in range(NC):\n     yes=self.y==c\n     score=np.nan_to_num(chi2(self.a,yes)[0],nan=0,posinf=0,neginf=0)\n     high=np.asarray(self.a[yes].mean(axis=0)).ravel()>np.asarray(self.a[~yes].mean(axis=0)).ravel()\n     score[~high|numeric]=0\n     orders.append(np.argsort(-score,kind='stable')[:min(k,int((score>0).sum()))])\n    order=[]\n    for rank in range(max(map(len,orders))):\n     for o in orders:\n      if rank<len(o):order.append(int(o[rank]))\n    order+=list(np.argsort(-global_score,kind='stable'))\n   else:order=list(np.argsort(-global_score,kind='stable'))\n  z=self.a.tocsc();chosen=list(np.flatnonzero(numeric));seen=set(chosen);signatures=set()\n  for j in order:\n   j=int(j)\n   if j in seen:continue\n   seen.add(j)\n   if global_score[j]<=0:continue\n   lo,hi=z.indptr[j:j+2]\n   sig=hashlib.sha256(z.indices[lo:hi].tobytes()+z.data[lo:hi].tobytes()).digest()\n   if sig in signatures:continue\n   signatures.add(sig);chosen.append(j)\n   if len(chosen)>=k:break\n  self.cache[(method,k)]=np.sort(chosen);return self.cache[(method,k)]\n\nclass SpecificityPost:\n def fit(self,a,names,target):\n  self.names=np.array(names);self.binary=np.flatnonzero([not n.startswith(('numeric:','module:')) for n in names]);self.summary=np.flatnonzero([n.startswith(('numeric:','module:')) for n in names])\n  z=(a[:,self.binary]!=0).astype(np.float32)\n  counts=np.vstack([np.asarray(z[target==c].sum(axis=0)).ravel() for c in range(NC)])\n  prior=np.bincount(target,minlength=NC)/len(target);freq=counts.sum(axis=0)\n  posterior=(counts+10*prior[:,None])/(freq+10)\n  kl=(posterior*np.log(np.maximum(posterior,1e-12)/prior[:,None])).sum(axis=0)\n  self.weights=(.25+np.sqrt(np.maximum(kl,0))*freq/(freq+20)).astype(np.float32)\n  self.scaler=MaxAbsScaler().fit(a[:,self.summary]);return self\n def transform(self,a):\n  from sklearn.preprocessing import normalize\n  block=normalize(a[:,self.binary].multiply(self.weights),norm='l2')\n  return csr32(sparse.hstack([block,self.scaler.transform(a[:,self.summary])],format='csr'))\n\nNEW={}\nfor view in ['raw','gated_syn_down_s3']:\n for method in ['global','balanced']:\n  NEW[f'{view}_{method}_lgb4096']=dict(view=view,algo='lgb',k=4096,method=method)\nfor c in [.1,.5]:NEW[f'balanced_lr_C{c}']=dict(view='raw',algo='lr',k=2048,method='balanced',C=c)\nfor c in [1.,4.]:NEW[f'specificity_lr_C{c}']=dict(view='raw',algo='lr',k=4096,method='balanced',C=c,specificity=True)\n\ndef fit_new6(name,idx,cache):\n cfg=NEW[name];view=cfg['view']\n if view not in cache:\n  enc,space,a,_=prepare_view(idx,view);cache[view]=(enc,space,a,NewSelector(a,yi[idx],space.names))\n enc,space,a,sel=cache[view];cols=sel.select(cfg['method'],cfg['k']);target=yi[idx]\n if cfg.get('specificity'):post=SpecificityPost().fit(a[:,cols],space.names[cols],target)\n else:post=ColumnPost().fit(a[:,cols],space.names[cols])\n est=estimator(cfg);tx=post.transform(a[:,cols])\n if cfg.get('augment'):\n  if '_aug' not in cache:cache['_aug']=parse_events(rare_background(X.iloc[idx],[events[i] for i in idx],SEED+777,count=10))\n  ar=enc.transform(cache['_aug'],exclude_events=[events[i] for i in idx]);ax=post.transform(space.transform(ar)[:,cols])\n  tx=csr32(sparse.vstack([tx,ax],format='csr'));target=np.concatenate([target,target]);rw=np.r_[np.ones(len(idx)),np.full(len(idx),.5)]\n else:rw=np.ones(len(idx))\n if cfg['algo']=='xgb':est.fit(tx,target,sample_weight=compute_sample_weight('balanced',target)*rw)\n elif cfg.get('augment'):est.fit(tx,target,sample_weight=rw)\n else:est.fit(tx,target)\n m=Member(name,view,enc,space,cols,post,est)\n # Column order of a specificity block changes, but raw_names always describe the vectorizer input.\n if cfg.get('specificity'):m.names=np.concatenate([m.raw_names[post.binary],m.raw_names[post.summary]])\n return m\n(OUT/'new_configurations.json').write_text(json.dumps(NEW,indent=2))\nprint('New configurations',len(NEW))\n\nNEW.update({'gated_s3_aug_global_lgb4096': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 4096, 'method': 'global', 'augment': True}, 'gated_s3_aug_balanced_lgb4096': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 4096, 'method': 'balanced', 'augment': True}, 'raw_aug_global_xgb4096_d3': {'view': 'raw', 'algo': 'xgb', 'k': 4096, 'method': 'global', 'depth': 3, 'augment': True}, 'raw_aug_global_xgb4096_d5': {'view': 'raw', 'algo': 'xgb', 'k': 4096, 'method': 'global', 'depth': 5, 'augment': True}})\n(OUT/'new_configurations.json').write_text(json.dumps(NEW,indent=2))\n\nprint('Total new configurations including augmentation extension:',len(NEW))\n", "def fit_plan6(idx,plan,basefit=None):\n ms=[];ws=[];cache={}\n for name,w in plan:\n  if name=='v5_reference':\n   old,ow=basefit if basefit is not None else fit_base(idx)\n   ms.extend(old);ws.extend(ow*w)\n  else:ms.append(fit_new6(name,idx,cache));ws.append(w)\n return ms,np.array(ws)\n\nV6_PLAN=[('v5_reference',.75),('gated_s3_aug_global_lgb4096',.25)]\ndef fit_v6(idx):return fit_plan6(idx,V6_PLAN)\n", "from sklearn.svm import LinearSVC\nfrom sklearn.naive_bayes import ComplementNB\nfrom sklearn.feature_selection import SelectKBest\n\ndef expand10(raw):\n t=canon(raw);m=re.fullmatch(r'([0-9]+)_([0-9]+)([A-Z]+)>([A-Z]+)',t)\n if m and int(m[2])-int(m[1])+1==len(m[3])==len(m[4]):\n  return tuple(f'{a}{int(m[1])+j}{b}' for j,(a,b) in enumerate(zip(m[3],m[4])))\n return (t,)\n\ndef kind10(t):\n if 'fs' in t:return 'frameshift'\n if t.endswith('*'):return 'stop'\n if 'delins' in t:return 'delins'\n if 'del' in t:return 'deletion'\n if 'ins' in t:return 'insertion'\n m=re.fullmatch(r'([A-Z])[0-9]+([A-Z])',t)\n if m:return 'synonymous' if m[1]==m[2] else 'substitution'\n return 'other'\n\ndef packet10(frame,ev=None):\n if ev is None:ev=parse_events(frame[genes])\n resolved=[[(g,tuple(sorted({s for t in ts for s in expand10(t)}))) for g,ts in row] for row in ev]\n missing=frame[genes].isna().to_numpy()\n return dict(events=resolved,missing=missing)\n\ndef records10(packet,representation,numeric=True):\n rs=[]\n for i,row in enumerate(packet['events']):\n  d={};syn=nonsyn=structural=unparsed=0\n  for g,ts in row:\n   d['gene:'+g]=1.;types={kind10(t) for t in ts}\n   sy='synonymous' in types;ns=bool(types.intersection({'substitution','stop','frameshift','deletion','insertion','delins'}));st=bool(types.intersection({'stop','frameshift','deletion','insertion','delins'}));ot='other' in types\n   syn+=sy;nonsyn+=ns;structural+=st;unparsed+=ot\n   if representation=='type':\n    for typ in types:d['type:'+g+':'+typ]=1.\n   elif representation=='functional':\n    for flag,family in [(sy,'syn_gene'),(ns,'nonsyn_gene'),(st,'structural_gene'),(ot,'unparsed_gene')]:\n     if flag:d[family+':'+g]=1.\n  if numeric:\n   d['numeric:log_observed_mutated_genes']=np.log1p(len(row));d['numeric:log_missing_genes']=np.log1p(int(packet['missing'][i].sum()))\n   if representation!='gene':\n    for title,value in [('syn',syn),('nonsyn',nonsyn),('structural',structural),('unparsed',unparsed)]:\n     d['numeric:log_'+title+'_genes']=np.log1p(value);d['numeric:fraction_'+title+'_genes']=value/max(len(row),1)\n  rs.append(d)\n return rs\n\ndef mask_packet10(packet,probability,seed):\n rng=np.random.default_rng(seed);out=[];missing=packet['missing'].copy();gpos={g:j for j,g in enumerate(genes)}\n for i,row in enumerate(packet['events']):\n  keep=[]\n  for g,ts in row:\n   if rng.random()<probability:missing[i,gpos[g]]=True\n   else:keep.append((g,ts))\n  out.append(keep)\n return dict(events=out,missing=missing)\n\ndef select_cols10(a,target,k):\n if k is None or a.shape[1]<=k:return np.arange(a.shape[1])\n with warnings.catch_warnings():\n  warnings.simplefilter('ignore',RuntimeWarning);scores=np.nan_to_num(chi2(a,target)[0],nan=0,posinf=0,neginf=0)\n return np.sort(np.argsort(-scores,kind='stable')[:k])\n\nclass MissingBernoulli10:\n def __init__(self,alpha=.1,pooled=False,fit_prior=False,k=None):self.alpha=alpha;self.pooled=pooled;self.fit_prior=fit_prior;self.k=k\n def fit(self,a,target,missing,sample_weight=None):\n  self.cols=select_cols10(a,target,self.k);a=csr32(a[:,self.cols]);missing=csr32(missing[:,self.cols]);w=np.ones(len(target)) if sample_weight is None else np.asarray(sample_weight)\n  self.classes_=np.unique(target);nclass=np.array([w[target==c].sum() for c in self.classes_])\n  counts=np.vstack([np.asarray(a[target==c].multiply(w[target==c,None]).sum(axis=0)).ravel() for c in self.classes_])\n  absent=np.vstack([np.asarray(missing[target==c].multiply(w[target==c,None]).sum(axis=0)).ravel() for c in self.classes_])\n  observed=nclass[:,None]-absent\n  if self.pooled:\n   denominator=w.sum()-np.asarray(missing.multiply(w[:,None]).sum(axis=0)).ravel()\n   prior=np.clip(np.asarray(a.multiply(w[:,None]).sum(axis=0)).ravel()/np.maximum(denominator,1e-9),1e-7,1-1e-7)\n   prob=(counts+self.alpha*prior)/(observed+self.alpha)\n  else:prob=(counts+self.alpha)/(observed+2*self.alpha)\n  prob=np.clip(prob,1e-7,1-1e-7);self.log0=np.log1p(-prob);self.logit=np.log(prob)-self.log0\n  prior=nclass/nclass.sum() if self.fit_prior else np.full(len(nclass),1/len(nclass));self.logprior=np.log(prior)\n  self.coef_=self.logit;return self\n def predict_proba(self,a,missing):\n  a=csr32(a[:,self.cols]);missing=csr32(missing[:,self.cols])\n  score=np.asarray(a@self.logit.T)+self.log0.sum(axis=1)+self.logprior-np.asarray(missing@self.log0.T)\n  return softmax(score,axis=1)\n\nclass GroupSigmoidSVM10:\n def __init__(self,C=.1,k=2048,balanced_calibration=False):self.C=C;self.k=k;self.balanced_calibration=balanced_calibration\n def fit(self,a,target,profile_groups,sample_weight):\n  self.classes_=np.unique(target);oof=np.zeros((len(target),len(self.classes_)),float)\n  for tr,va in StratifiedGroupKFold(3,shuffle=True,random_state=310).split(a,target,profile_groups):\n   assert not set(profile_groups[tr]).intersection(profile_groups[va])\n   cols=select_cols10(a[tr],target[tr],self.k);scaler=MaxAbsScaler().fit(a[tr][:,cols])\n   est=LinearSVC(C=self.C,class_weight='balanced',dual='auto',max_iter=5000,random_state=42).fit(csr32(scaler.transform(a[tr][:,cols])),target[tr],sample_weight=sample_weight[tr])\n   scores=est.decision_function(csr32(scaler.transform(a[va][:,cols])))\n   assert np.array_equal(est.classes_,self.classes_);oof[va]=scores\n  self.calibrators=[]\n  for j,c in enumerate(self.classes_):\n   cal=LogisticRegression(C=1.,class_weight='balanced' if self.balanced_calibration else None,max_iter=1000).fit(oof[:,j,None],(target==c).astype(int),sample_weight=sample_weight);self.calibrators.append(cal)\n  self.cols=select_cols10(a,target,self.k);self.scaler=MaxAbsScaler().fit(a[:,self.cols]);self.est=LinearSVC(C=self.C,class_weight='balanced',dual='auto',max_iter=5000,random_state=42).fit(csr32(self.scaler.transform(a[:,self.cols])),target,sample_weight=sample_weight);self.coef_=self.est.coef_;return self\n def predict_proba(self,a):\n  score=self.est.decision_function(csr32(self.scaler.transform(a[:,self.cols])))\n  p=np.column_stack([cal.predict_proba(score[:,j,None])[:,1] for j,cal in enumerate(self.calibrators)])\n  return p/np.maximum(p.sum(axis=1,keepdims=True),1e-12)\n\nclass Member10:\n def __init__(self,name,space,est,representation,numeric,cols=None,scaler=None):\n  self.name=name;self.view='v10_'+representation;self.representation=representation;self.numeric=numeric;self.vec=space.compact_vec(np.arange(len(space.names)));self.est=est;self.cols=est.cols if cols is None else cols;self.raw_names=space.names[self.cols];self.names=self.raw_names;self.scaler=scaler\n  lookup={g:i for i,g in enumerate(genes)};self.feature_gene_positions=np.array([lookup.get(str(s).split(':')[1],-1) if ':' in s and not str(s).startswith('numeric:') else -1 for s in space.names])\n def missing_matrix(self,packet):\n  out=np.zeros((len(packet['events']),len(self.feature_gene_positions)),bool);valid=self.feature_gene_positions>=0\n  out[:,valid]=packet['missing'][:,self.feature_gene_positions[valid]]\n  return csr32(sparse.csr_matrix(out))\n def predict_packet(self,packet,records=None):\n  rs=records10(packet,self.representation,self.numeric) if records is None else records;a=csr32(self.vec.transform(rs))\n  if isinstance(self.est,MissingBernoulli10):return self.est.predict_proba(a,self.missing_matrix(packet))\n  if isinstance(self.est,GroupSigmoidSVM10):return self.est.predict_proba(a)\n  tx=a[:,self.cols]\n  if self.scaler is not None:tx=csr32(self.scaler.transform(tx))\n  return self.est.predict_proba(tx)\n def predict_frame(self,frame):return self.predict_packet(packet10(frame[genes]))\n\npredict_members6=predict_members\ndef predict_members(members,weights,frame):\n old=[(m,w) for m,w in zip(members,weights) if not hasattr(m,'predict_frame')];p=predict_members6([m for m,w in old],[w for m,w in old],frame) if old else None\n packet=None\n for m,w in zip(members,weights):\n  if hasattr(m,'predict_frame'):\n   if packet is None:packet=packet10(frame[genes])\n   q=m.predict_packet(packet)*w;p=q if p is None else p+q\n return p\n\nC10={\n 'gene_nb_a0.1':dict(rep='gene',algo='nb',alpha=.1,pooled=False,fit_prior=False,k=None,mask=0.),\n 'gene_nb_a1':dict(rep='gene',algo='nb',alpha=1.,pooled=False,fit_prior=False,k=None,mask=0.),\n 'gene_nb_empirical_prior':dict(rep='gene',algo='nb',alpha=.1,pooled=False,fit_prior=True,k=None,mask=0.),\n 'gene_nb_pooled20':dict(rep='gene',algo='nb',alpha=20.,pooled=True,fit_prior=False,k=None,mask=0.),\n 'type_nb_pooled20':dict(rep='type',algo='nb',alpha=20.,pooled=True,fit_prior=False,k=4096,mask=0.),\n 'gene_lr':dict(rep='gene',algo='lr',C=.1,k=2048,mask=0.),\n 'gene_lr_mask10':dict(rep='gene',algo='lr',C=.1,k=2048,mask=.1),\n 'gene_lr_mask25':dict(rep='gene',algo='lr',C=.1,k=2048,mask=.25),\n 'type_lr':dict(rep='type',algo='lr',C=.1,k=2048,mask=0.),\n 'type_lr_mask10':dict(rep='type',algo='lr',C=.1,k=2048,mask=.1),\n 'functional_lr':dict(rep='functional',algo='lr',C=.1,k=2048,mask=0.),\n 'functional_lr_mask10':dict(rep='functional',algo='lr',C=.1,k=2048,mask=.1),\n 'gene_svm':dict(rep='gene',algo='svm',C=.1,k=2048,mask=0.),\n 'type_svm':dict(rep='type',algo='svm',C=.1,k=2048,mask=0.),\n 'functional_svm':dict(rep='functional',algo='svm',C=.1,k=2048,mask=0.),\n 'functional_svm_mask10':dict(rep='functional',algo='svm',C=.1,k=2048,mask=.1),\n 'functional_lgb':dict(rep='functional',algo='lgb',k=2048,mask=0.),\n 'functional_lgb_mask10':dict(rep='functional',algo='lgb',k=2048,mask=.1),\n 'functional_lgb_mask25':dict(rep='functional',algo='lgb',k=2048,mask=.25)}\n\ndef prepare10(idx,representation,numeric,cache):\n key=(representation,numeric)\n if key not in cache:\n  if '_clean' not in cache:cache['_clean']=packet10(X.iloc[idx],[events[i] for i in idx])\n  rs=records10(cache['_clean'],representation,numeric);space=Space();a=space.fit(rs);cache[key]=(space,a)\n return cache[key]\n\ndef train10(name,idx,cache):\n cfg=C10[name];numeric=cfg['algo'] not in ['nb','cnb'];space,a=prepare10(idx,cfg['rep'],numeric,cache);target=yi[idx];rw=np.ones(len(idx));train_groups=groups[idx]\n augmented=None\n if cfg['mask']:\n  key=('masked',cfg['mask'])\n  if key not in cache:cache[key]=mask_packet10(cache['_clean'],cfg['mask'],1042+int(cfg['mask']*100))\n  augmented=cache[key];ar=records10(augmented,cfg['rep'],numeric);a_aug=space.transform(ar);a=csr32(sparse.vstack([a,a_aug],format='csr'));target=np.r_[target,target];train_groups=np.r_[train_groups,train_groups];rw=np.r_[rw,.5*rw]/1.5\n if cfg['algo']=='nb':\n  est=MissingBernoulli10(cfg['alpha'],cfg['pooled'],cfg['fit_prior'],cfg['k'])\n  temp=Member10(name,space,est,cfg['rep'],False,cols=np.arange(a.shape[1]));missing=temp.missing_matrix(cache['_clean']);est.fit(a,target,missing,sample_weight=rw)\n  return Member10(name,space,est,cfg['rep'],False)\n if cfg['algo']=='cnb':\n  cols=select_cols10(a,target,cfg['k']);est=ComplementNB(alpha=cfg['alpha'],norm=False).fit(a[:,cols],target,sample_weight=rw)\n  return Member10(name,space,est,cfg['rep'],False,cols=cols)\n if cfg['algo']=='svm':\n  est=GroupSigmoidSVM10(cfg['C'],cfg['k'],cfg.get('balanced_calibration',False)).fit(a,target,train_groups,rw);return Member10(name,space,est,cfg['rep'],True)\n # Outer selection and scaler are fit on the original training patients only.\n cols=select_cols10(prepare10(idx,cfg['rep'],numeric,cache)[1],yi[idx],cfg['k'])\n scaler=MaxAbsScaler().fit(prepare10(idx,cfg['rep'],numeric,cache)[1][:,cols]) if cfg['algo']=='lr' else None\n tx=a[:,cols];tx=csr32(scaler.transform(tx)) if scaler is not None else csr32(tx)\n est=LogisticRegression(C=cfg['C'],class_weight='balanced',max_iter=2000) if cfg['algo']=='lr' else LGBMClassifier(n_estimators=300,learning_rate=.04,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=42)\n est.fit(tx,target,sample_weight=rw);return Member10(name,space,est,cfg['rep'],True,cols=cols,scaler=scaler)\n\n# A missing feature must contribute no Bernoulli presence or absence evidence.\ntoy=sparse.csr_matrix([[1,0],[1,0],[0,1],[0,1]],dtype=np.float32);toy_m=sparse.csr_matrix(toy.shape,dtype=np.float32)\nnbtoy=MissingBernoulli10(alpha=1.).fit(toy,np.array([0,0,1,1]),toy_m)\nassert np.allclose(nbtoy.predict_proba(sparse.csr_matrix([[1,0]],dtype=np.float32),sparse.csr_matrix([[0,0]],dtype=np.float32)),[[.9,.1]])\nassert np.allclose(nbtoy.predict_proba(sparse.csr_matrix([[0,0]],dtype=np.float32),sparse.csr_matrix([[1,1]],dtype=np.float32)),[[.5,.5]])\nassert all(kind10(t)=='synonymous' for t in expand10('1_2FS>FS'))\n(OUT/'configurations_v10.json').write_text(json.dumps(C10,indent=2));print('V10 candidates:',len(C10),'semantic checks passed')\n\nC10.update({'functional_svm_balanced': {'rep': 'functional', 'algo': 'svm', 'C': 0.1, 'k': 2048, 'mask': 0.0, 'balanced_calibration': True}, 'functional_svm_balanced_mask10': {'rep': 'functional', 'algo': 'svm', 'C': 0.1, 'k': 2048, 'mask': 0.1, 'balanced_calibration': True}, 'type_complement_nb_a0.1': {'rep': 'type', 'algo': 'cnb', 'alpha': 0.1, 'k': 4096, 'mask': 0.0}, 'type_complement_nb_a1': {'rep': 'type', 'algo': 'cnb', 'alpha': 1.0, 'k': 4096, 'mask': 0.0}, 'functional_complement_nb_a1': {'rep': 'functional', 'algo': 'cnb', 'alpha': 1.0, 'k': 4096, 'mask': 0.0}})\n(OUT/'configurations_v10.json').write_text(json.dumps(C10,indent=2))\nprint('Extended candidates:',len(C10))\n", "# P43: HGVS 2016 principles; this is a conservative dataset parser, not a full HGVS validator.\nAA16=set('ACDEFGHIKLMNPQRSTVWY');THREE16=dict(zip(['Ala','Arg','Asn','Asp','Cys','Gln','Glu','Gly','His','Ile','Leu','Lys','Met','Phe','Pro','Ser','Thr','Trp','Tyr','Val','Ter','Xaa'],list('ARNDCQEGHILKMFPSTWYV')+['*','X']))\n# Each event: key, kind, position, components(position,ref,alt,kind), flags, fs_stop_distance.\ndef parse_token16(raw):\n s=str(raw).strip();core=s[2:] if s.startswith('p.') else s;uncertain=core.startswith('(') and core.endswith(')');core=core[1:-1] if uncertain else core;core=normalize_token25(core)\n if re.match('^(?:'+'|'.join(THREE16)+')[0-9]+',core):core=re.sub('|'.join(THREE16),lambda m:THREE16[m[0]],core)\n key=('uncertain|' if uncertain else '')+core;flags=['uncertain'] if uncertain else [];pos=None;parts=();dist=None;kind='unparsed'\n def effect(a,b):\n  if a=='*' and b=='*':return 'stop_retained'\n  if a=='*':return 'stop_lost' if b in AA16 else 'ambiguous_residue'\n  if b=='*':return 'stop_gained' if a in AA16 else 'ambiguous_residue'\n  if a not in AA16 or b not in AA16:return 'ambiguous_residue'\n  return 'synonymous' if a==b else 'missense'\n m=re.fullmatch(r'([A-Z*])([1-9][0-9]*)([A-Z*])',core)\n if m:\n  a,p,b=m.groups();pos=int(p);kind=effect(a,b);parts=((pos,a,b,kind),)\n  if 'X' in [a,b]:flags.append('ambiguous_X')\n else:\n  m=re.fullmatch(r'([1-9][0-9]*)_([1-9][0-9]*)([A-Z*]+)>([A-Z*]+)',core)\n  if m:\n   start,end,ref,alt=m.groups();start=int(start);end=int(end);pos=start;span=end-start+1\n   if span>0 and span==len(ref)==len(alt) and all(a in AA16 or a=='*' for a in ref+alt):\n    parts=tuple((start+j,a,b,effect(a,b)) for j,(a,b) in enumerate(zip(ref,alt)));kind='synonymous_block' if all(a==b for a,b in zip(ref,alt)) else 'block_change';flags.append('resolved_equal_length_block')\n   else:kind='complex_range';flags.append('unresolved_range')\n  elif 'fs' in core:\n   kind='frameshift';m=re.match(r'([A-Z*]+)([1-9][0-9]*)',core);pos=int(m[2]) if m else None;m=re.search(r'fs(?:\\*|Ter)([1-9][0-9]*)$',core);dist=int(m[1]) if m else None\n  elif 'delins' in core:kind='delins'\n  elif 'del' in core:kind='deletion'\n  elif 'ins' in core:kind='insertion'\n  elif 'dup' in core:kind='duplication'\n  elif 'ext' in core:kind='extension'\n  elif core=='=':kind='no_change_unspecified'\n  elif re.fullmatch(r'[A-Z*]+[1-9][0-9]*',core):kind='incomplete';flags.append('missing_alternate')\n  if pos is None:\n   # Protein coordinate only: do not mistake fs termination distance for the affected residue.\n   m=re.match(r'(?:[A-Z*]+)?([1-9][0-9]*)',core);pos=int(m[1]) if m else None\n return (key,kind,pos,parts,tuple(flags),dist)\n\ndef parse_rows16(frame):\n cols=list(frame.columns);cache={};rows=[]\n for row in frame.itertuples(index=False,name=None):\n  mutations=[];missing=[];raw_n=0;duplicates=0\n  for gene,value in zip(cols,row):\n   if value is None or pd.isna(value) or (isinstance(value,str) and not value.strip()):missing.append(gene);continue\n   if str(value).strip()=='WT':continue\n   tokens=str(value).split();raw_n+=len(tokens);tokens=[normalize_token25(t) for t in tokens];unique=list(dict.fromkeys(tokens));duplicates+=len(tokens)-len(unique);events=[]\n   for token in unique:\n    if token not in cache:cache[token]=parse_token16(token)\n    events.append(cache[token])\n   mutations.append((gene,tuple(events)))\n  rows.append({'mutations':tuple(mutations),'missing':tuple(missing),'raw_tokens':raw_n,'duplicate_tokens':duplicates})\n return rows\n\nFIXTURES16=[('R132H','missense',132),('R132R','synonymous',132),('R213*','stop_gained',213),('*363*','stop_retained',363),('*213Q','stop_lost',213),('K16Rfs*23','frameshift',16),('1_2FS>FS','synonymous_block',1),('483_484MP>IA','block_change',483),('10_12AAA>DD','complex_range',10),('490del','deletion',490),('p.(Arg123His)','missense',123),('R123X','stop_gained',123),('A391','incomplete',391),('FS','unparsed',None),('G12delinsAA','delins',12)]\nfor token,expected_kind16,expected_pos16 in FIXTURES16:\n result=parse_token16(token);assert result[1]==expected_kind16 and result[2]==expected_pos16,(token,result)\nassert parse_token16('K16Rfs*23')[5]==23\nassert parse_token16('1_2FS>FS')[3]==((1,'F','F','synonymous'),(2,'S','S','synonymous'))\nassert 'uncertain' in parse_token16('p.(Arg123His)')[4]\nassert parse_rows16(pd.DataFrame({'G':['WT',None,'R1H R1H']}))[1]['missing']==('G',)\nassert parse_rows16(pd.DataFrame({'G':['WT',None,'R1H R1H']}))[2]['duplicate_tokens']==1\nprint('P43 annotation safety fixtures passed:',len(FIXTURES16),flush=True)\n# P25-P40: small, explicit gene subsets in the publications, not whole pathways or diagnostic rules.\nPANELS16={\n 'P25_breast':('TP53','PIK3CA','GATA3','MAP3K1'),\n 'P26_colorectal':('APC','TP53','SMAD4','PIK3CA','KRAS','ARID1A','SOX9','FAM123B','POLE','MLH1'),\n 'P27_glioma':('IDH1','IDH2','TP53','ATRX','CIC','FUBP1','NOTCH1'),\n 'P29_gbm':('EGFR','PDGFRA'),\n 'P30_renal':('VHL','PBRM1','SETD2','ARID1A','SMARCA4'),\n 'P31_papillary_renal':('MET','SETD2','FH'),\n 'P32_ovarian':('TP53','NF1','BRCA1','BRCA2','RB1','CDK12'),\n 'P33_uterine':('TP53','PTEN','CTNNB1','PIK3CA','ARID1A','KRAS','ARID5B','POLE'),\n 'P34_thyroid':('BRAF','EIF1AX','PPM1D','CHEK2'),\n 'P35_prostate':('SPOP','FOXA1','IDH1'),\n 'P36_lung_ad':('RIT1','MGA','EGFR','RBM10','NF1','MET','ERBB2'),\n 'P37_lung_scc':('TP53','HLA-A','NFE2L2','KEAP1','CDKN2A','RB1'),\n 'P38_melanoma':('BRAF','NRAS','HRAS','KRAS','NF1','KIT'),\n 'P39_gastric':('PIK3CA','RHOA'),\n 'P40_headneck':('PIK3CA','TRAF3','TP53','CDKN2A','HRAS','CASP8','NOTCH1','NSD1','AJUBA','FAT1','NFE2L2')}\nprint('Literature panel subsets:',len(PANELS16),'(external biological prior; no external patient data)',flush=True)\n", "from sklearn.feature_extraction import DictVectorizer\nfrom sklearn.feature_selection import chi2\nfrom sklearn.preprocessing import MaxAbsScaler\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.ensemble import ExtraTreesClassifier\nfrom scipy import sparse\nfrom collections import Counter\nimport copy\nVIEWS16=['legacy_raw','hierarchical','rare_backoff','rare_backoff_panels']\n# Literature-derived panel snapshots are explicit priors, not learned class labels.\nclass Preprocess16:\n def __init__(self,view,min_df=3,k=4096):self.view=view;self.min_df=min_df;self.k=k\n def fit(self,rows,target):\n  self.df=Counter((g,e[0]) for row in rows for g,evs in row['mutations'] for e in {e[0]:e for e in evs}.values())\n  records=self.records(rows,training=True);self.vec=DictVectorizer(dtype=np.float32);a=self.vec.fit_transform(records);names=self.vec.get_feature_names_out();support=np.asarray((a!=0).sum(0)).ravel();keep=(support>=self.min_df)|np.array([s.startswith(('num:','module:','numeric:','panel:')) for s in names]);self.keep=np.flatnonzero(keep);a=a[:,self.keep]\n  with warnings.catch_warnings():\n   warnings.simplefilter('ignore');scores=np.nan_to_num(chi2(a,target)[0],nan=0,posinf=0,neginf=0)\n  self.cols=np.sort(np.argsort(-scores,kind='stable')[:min(self.k,a.shape[1])]);self.names=names[self.keep[self.cols]];selected=a[:,self.cols];self.scale=MaxAbsScaler().fit(selected);self.residual_weights=np.ones(len(self.names),np.float32)\n  if self.view.startswith('rare_backoff'):\n   for j,name in enumerate(self.names):\n    if name.startswith('variant:'):\n     _,gene,key=name.split(':',2);n=self.df.get((gene,key),0);self.residual_weights[j]=n/(n+5.)\n  return self\n def records(self,rows,training=False):\n  # legacy records were computed pointwise using the frozen v6 raw encoder.\n  if self.view=='legacy_raw':return [r['legacy'] for r in rows]\n  result=[]\n  for row in rows:\n   d={};types=Counter();gene_effects={};n=0;rare=0;new=0;uncertain=0;incomplete=0;components=0;ambiguous_x=0\n   for g,evs in row['mutations']:\n    evs=list({e[0]:e for e in evs}.values());d['gene:'+g]=1.;d['gene_event_count:'+g]=np.log1p(len(evs));local=Counter();bin64=set();bin256=set()\n    for key,kind,pos,parts,flags,fsdist in evs:\n     n+=1;support=max(0,self.df.get((g,key),0)-(int(training)));israre=support<3;rare+=israre;new+=support==0;uncertain+='uncertain' in flags;incomplete+=kind in ['incomplete','unparsed','complex_range'];ambiguous_x+='ambiguous_X' in flags\n     effects=[p[3] for p in parts] or [kind];types.update(effects);local.update(effects);d['format:'+g+':'+kind]=1.;components+=len(parts)\n     if self.df.get((g,key),0)>=3:\n      d['variant:'+g+':'+key]=1.\n     if self.view.startswith('rare_backoff'):\n      bucket='rare' if israre else 'supported';d[bucket+'_gene:'+g]=d.get(bucket+'_gene:'+g,0)+1\n     for p,a,b,effect in parts:\n      d['aa_meta:'+g+':'+a+'>'+b]=d.get('aa_meta:'+g+':'+a+'>'+b,0)+1;d['aa_global:'+a+'>'+b]=d.get('aa_global:'+a+'>'+b,0)+1\n     if pos is not None:bin64.add((pos-1)//64);bin256.add((pos-1)//256)\n     # Absolute protein bins are an engineering heuristic, not genomic bins or biological domains.\n     if fsdist is not None:d['fs_stop_distance:'+g]=np.log1p(fsdist)\n    for kind,count in local.items():d['effect:'+g+':'+kind]=np.log1p(count)\n    for b in bin64:d[f'protein_bin64:{g}:{b}']=1.\n    for b in bin256:d[f'protein_bin256:{g}:{b}']=1.\n    gene_effects[g]=local\n   for name,count in [('observed_genes',len(row['mutations'])),('unique_events',n),('raw_tokens',row['raw_tokens']),('duplicate_tokens',row['duplicate_tokens']),('missing_genes',len(row['missing'])),('uncertain_events',uncertain),('incomplete_events',incomplete),('ambiguous_X',ambiguous_x),('resolved_components',components)]:d['num:log_'+name]=np.log1p(count)\n   for kind in ['synonymous','missense','stop_gained','stop_lost','stop_retained','frameshift','deletion','insertion','delins','complex_range','incomplete','unparsed','ambiguous_residue']:\n    d['num:log_'+kind]=np.log1p(types[kind]);d['num:fraction_'+kind]=types[kind]/max(sum(types.values()),1)\n   if self.view.startswith('rare_backoff'):d['num:log_rare_events']=np.log1p(rare);d['num:rare_fraction']=rare/max(n,1);d['num:log_unseen_events']=np.log1p(new)\n   if self.view.endswith('panels'):\n    observed=set(gene_effects)\n    for panel,gs in PANELS16.items():\n     count=len(observed.intersection(gs));d['panel:'+panel+':mutated_genes']=np.log1p(count)\n     flags=sum(bool(set(gene_effects.get(g,{})).intersection({'stop_gained','frameshift','deletion','delins'})) for g in gs);d['panel:'+panel+':truncation_notation_genes']=np.log1p(flags)\n   result.append(d)\n  return result\n def transform(self,rows,training=False):\n  a=self.vec.transform(self.records(rows,training=training));a=a[:,self.keep][:,self.cols];return sparse.csr_matrix(self.scale.transform(a).multiply(self.residual_weights),dtype=np.float32)\n\nclass Model16:\n def __init__(self,view,algo):self.view=view;self.algo=algo\n def fit(self,rows,target):\n  self.pre=Preprocess16(self.view).fit(rows,target);a=self.pre.transform(rows,training=True)\n  self.est=LogisticRegression(C=.1,class_weight='balanced',max_iter=1200) if self.algo=='lr' else ExtraTreesClassifier(n_estimators=200,max_depth=20,min_samples_leaf=2,max_features='sqrt',class_weight='balanced',n_jobs=2,random_state=16016)\n  self.est.fit(a,target);return self\n def predict_proba(self,rows):\n  p=self.est.predict_proba(self.pre.transform(rows));out=np.zeros((len(rows),NC),np.float64);out[:,self.est.classes_.astype(int)]=p;return out\n def predict_frame(self,frame):\n  rows=parse_rows16(frame[GENES16]);legacy=EvidenceEncoder(1.,1.).fit([]).transform(parse_events(frame[GENES16]))\n  for row,d in zip(rows,legacy):row['legacy']=d\n  return self.predict_proba(rows)\n", "# Protocol frozen before new outer results. The old holdout is never scored.\nfrom sklearn.metrics import f1_score\nfrom sklearn.utils.class_weight import compute_sample_weight\nimport gc\nGENES16=list(genes)\nparsed17=parse_rows16(X)\nraw17=EvidenceEncoder(1.,1.).fit([]).transform(parse_events(X))\nfor row,record in zip(parsed17,raw17):row['legacy']=record\n# Identical canonical and raw profiles stay in the same group.\nparent17=np.arange(len(train))\ndef root17(i):\n while parent17[i]!=i:parent17[i]=parent17[parent17[i]];i=parent17[i]\n return i\ndef join17(a,b):parent17[root17(b)]=root17(a)\ncanonical17=[hashlib.sha256(repr((tuple(sorted((g,tuple(sorted(set(e[0] for e in ev)))) for g,ev in row['mutations'])),tuple(sorted(row['missing'])))).encode()).hexdigest() for row in parsed17]\nfor identifiers in [groups,canonical17]:\n seen17={}\n for i,key in enumerate(identifiers):\n  if key in seen17:join17(seen17[key],i)\n  else:seen17[key]=i\nG17=np.array([root17(i) for i in range(len(train))]);overlap17=set(G17[dev])&set(G17[hold]);D17=dev[~np.isin(G17[dev],list(overlap17))]\nassert not set(G17[D17])&set(G17[hold])\nPLANS17={'v10_reference':(1.,0.,0.),'add_lr20':(.8,.2,0.),'add_lgb20':(.8,0.,.2),'add_both40':(.6,.2,.2)}\nPROTOCOL17={'version':17,'outer_seed':17017,'outer_folds':3,'inner_seed_base':17018,'inner_folds':2,'plans':PLANS17,'new_view':'rare_backoff','k':8192,'min_df':3,'lr_C':1.,'lgb':{'n_estimators':300,'num_leaves':15,'max_depth':5,'learning_rate':.04,'reg_lambda':10.,'min_child_samples':20},'selection':'inner mean Macro F1; baseline wins exact tie','test_opened':False,'holdout_scored':False,'adoption_gate':'selected-v10 mean >=0.003 and every outer delta>=0; else experimental only','historical_development_bias':'same development cohort used before; new seeds do not make external validation'}\nSHA17=hashlib.sha256((DATA/'train.csv').read_bytes()).hexdigest()\n# Frozen baseline parameters are embedded; no baseline search.\ndef f17(target,p):return float(f1_score(target,p.argmax(1),average='macro',labels=np.arange(NC),zero_division=0))\nclass Rich17:\n def __init__(self,algo):self.algo=algo\n def fit(self,rows,target):\n  self.pre=Preprocess16('rare_backoff',min_df=3,k=8192).fit(rows,target);a=self.pre.transform(rows,training=True)\n  if self.algo=='lr':self.est=LogisticRegression(C=1.,class_weight='balanced',max_iter=1600)\n  else:self.est=LGBMClassifier(n_estimators=300,num_leaves=15,max_depth=5,learning_rate=.04,reg_lambda=10.,min_child_samples=20,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=17017)\n  self.est.fit(a,target);return self\n def predict_proba(self,rows):\n  p=self.est.predict_proba(self.pre.transform(rows));out=np.zeros((len(rows),NC));out[:,self.est.classes_.astype(int)]=p;return out\n def predict_frame(self,frame):return self.predict_proba(parse_rows16(frame[GENES16]))\nclass Ensemble17:\n def __init__(self,old,weights,nb,rich,plan):self.old=old;self.weights=weights;self.nb=nb;self.rich=rich;self.plan=plan\n def predict_frame(self,frame):\n  w=PLANS17[self.plan];p=w[0]*(.75*predict_members(self.old,self.weights,frame)+.25*self.nb.predict_frame(frame))\n  for key,weight in [('lr',w[1]),('lgb',w[2])]:\n   if weight:p+=weight*self.rich[key].predict_frame(frame)\n  return p\n\n", "from sklearn.naive_bayes import ComplementNB,BernoulliNB\nimport gc\n# Separate follow-up hypothesis after v18 first-fold result: counts and absence, not another tree.\nPLANS19={'v17_reference':(1.,0.,0.),'cnb10':(.9,.1,0.),'bnb10':(.9,0.,.1),'both20':(.8,.1,.1)}\nPROTOCOL19={'version':19,'outer_seed':17017,'outer_folds':3,'inner_seed_base':17018,'inner_folds':2,'confirmation_seed':19029,'confirmation_folds':3,'plans':PLANS19,'baseline':'fixed v17 add_both40','view':'rare_backoff','k':8192,'min_df':3,'cnb':{'alpha':1.,'norm':False},'bnb':{'alpha':1.,'binarize':0.,'fit_prior':False},'probability_temperature':2.,'primary_gate':'nested mean delta>=0.003 and each fold>=0','confirmation_gate':'fixed-plan mean delta>=0.003 and each fold>=0','test_opened':False,'holdout_scored':False,'historical_bias':'adaptive follow-up after v18 first-fold result; previous primary split already observed; never pool selection with v18 outer maxima'}\n# Frozen v19 definitions only; no new v19 search.\nclass New19:\n def __init__(self,algo):self.algo=algo\n def fit(self,rows,target):\n  self.pre=Preprocess16('rare_backoff',min_df=3,k=8192).fit(rows,target);a=self.pre.transform(rows,training=True)\n  self.est=ComplementNB(alpha=1.,norm=False) if self.algo=='cnb' else BernoulliNB(alpha=1.,binarize=0.,fit_prior=False)\n  self.est.fit(a,target);return self\n def predict_proba(self,rows):\n  p=softmax(self.est.predict_log_proba(self.pre.transform(rows))/2.,axis=1);out=np.zeros((len(rows),NC));out[:,self.est.classes_.astype(int)]=p;return out\n def predict_frame(self,frame):return self.predict_proba(parse_rows16(frame[GENES16]))\nclass Ensemble19:\n def __init__(self,baseline,models,plan):self.baseline=baseline;self.models=models;self.plan=plan\n def predict_frame(self,frame):\n  w=PLANS19[self.plan];p=w[0]*self.baseline.predict_frame(frame)\n  for name,weight in [('cnb',w[1]),('bnb',w[2])]:\n   if weight:p+=weight*self.models[name].predict_frame(frame)\n  return p\n\n", "from sklearn.neural_network import MLPClassifier\nfrom sklearn.utils.class_weight import compute_sample_weight\nimport gc\n\ndef corrupt23(rows):\n # Support and nuisance pool come from the current original training fold only, without labels.\n df=Counter((g,e[0]) for row in rows for g,ev in row['mutations'] for e in {e[0]:e for e in ev}.values());pool={(g,e[0]):(g,e) for row in rows for g,ev in row['mutations'] for e in ev if df[(g,e[0])]<=2 and e[1] in ['missense','synonymous','synonymous_block']};pool=list(pool.values());assert pool;rng=np.random.default_rng(23023);aug=[]\n for row in rows:\n  mutations=row['mutations'];keep=rng.random(len(mutations))>=.1\n  if len(keep) and not keep.any():keep[rng.integers(len(keep))]=True\n  retained={g:{e[0]:e for e in ev} for (g,ev),flag in zip(mutations,keep) if flag}\n  for _ in range(int(rng.poisson(20))):\n   g,e=pool[rng.integers(len(pool))]\n   if g in row['missing']:continue\n   retained.setdefault(g,{})[e[0]]=e\n  evs=tuple((g,tuple(events23.values())) for g,events23 in retained.items());aug.append({'mutations':evs,'missing':row['missing'],'raw_tokens':sum(len(ev) for g,ev in evs),'duplicate_tokens':0})\n return aug\nclass Model23:\n def __init__(self,algo):self.algo=algo\n def fit(self,rows,target):\n  self.pre=Preprocess16('rare_backoff',min_df=3,k=8192).fit(rows,target);a=self.pre.transform(rows,training=True);aug=corrupt23(rows);at=sparse.vstack([a,self.pre.transform(aug,training=False)],format='csr');yt=np.r_[target,target]\n  if self.algo=='ova':\n   self.est=LGBMClassifier(objective='multiclassova',is_unbalance=True,n_estimators=500,learning_rate=.04,num_leaves=15,max_depth=5,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,verbosity=-1,n_jobs=4,random_state=23023);sw=np.r_[np.ones(len(target)),.5*np.ones(len(target))]/1.5;self.est.fit(at,yt,sample_weight=sw)\n  else:\n   sw=compute_sample_weight('balanced',target);sw=np.r_[sw,.5*sw]/1.5;self.est=MLPClassifier(hidden_layer_sizes=(128,),alpha=1.,batch_size=128,learning_rate_init=.001,max_iter=180,early_stopping=False,random_state=23023);self.est.fit(at,yt,sample_weight=sw)\n  return self\n def predict_proba(self,rows):\n  p=np.asarray(self.est.predict_proba(self.pre.transform(rows)));p/=np.maximum(p.sum(1,keepdims=True),1e-12);out=np.zeros((len(rows),NC));out[:,self.est.classes_.astype(int)]=p;return out\n def predict_frame(self,frame):return self.predict_proba(parse_rows16(frame[GENES16]))\nPLANS23={'v19_reference':{'base':1.},'new_ova':{'ova':1.},'new_aug_mlp':{'mlp':1.},'new_average':{'ova':.5,'mlp':.5},'optional_ova20':{'base':.8,'ova':.2},'optional_mlp20':{'base':.8,'mlp':.2},'optional_both40':{'base':.6,'ova':.2,'mlp':.2}}\nPROTOCOL23={'version':23,'plans':PLANS23,'outer_seed':17017,'inner_seed_base':17018,'outer_folds':3,'inner_folds':2,'confirmation_seed':23033,'confirmation_folds':3,'preprocessor':'rare_backoff min_df3 k8192; only original fold fits vocabulary, DF, selection and scale','augmentation':{'mask_gene_prob':.1,'nuisance_poisson_mean':20,'nuisance_df_max':2,'nuisance_types':['missense','synonymous','synonymous_block'],'weight':.5,'transform_role':'inference DF on augmented rows; training LOO on originals','seed':23023},'fresh_noise_confirmation':{'poisson_mean':35,'seed_base':23034,'pool':'independent legacy rare_background mechanism, current training fold only','criterion':'mean delta>=-0.003'},'ova':{'loss':'multiclassova','is_unbalance':True,'trees':500,'leaves':15,'depth':5,'lambda':10},'mlp':{'hidden':[128],'alpha':1.,'epochs':180,'sample_weight':'balanced','early_stopping':False},'primary_gate':'mean delta>=0.005 and each fold>=-0.001','confirmation_gate':'mean delta>=0.003 and each fold>=-0.001 and paired pooled group bootstrap lower95>0 and fresh-noise35 mean delta>=-0.003','stress_gate':'historical rare-background20 and duplicate mean deltas>=-0.003','design_bias':'adaptive response to v22 historical stress decline; repeated development cohort/primary split. Fresh grouping, new draw and stronger noise are confirmation, not independent patients. No retuning on confirmation outcomes.','reference_public_user_reported':0.3664759128,'test_used_for_research':False,'old_holdout_scored':False}\n(OUT/'protocol.json').write_text(json.dumps(PROTOCOL23,indent=2))\nclass Export23:\n def __init__(self,models,plan):self.models=models;self.plan=plan;self.classes_=labels.copy();self.genes_=list(genes)\n def predict_frame(self,frame):\n  p=np.zeros((len(frame),len(self.classes_)))\n  for key,w in PLANS23[self.plan].items():p+=w*self.models[key].predict_frame(frame)\n  return p\n\ndef fit_old23(idx):\n ms,ws=fit_v6(idx);nb=train10('type_complement_nb_a0.1',idx,{});rich={key:Rich17(key).fit([parsed17[i] for i in idx],yi[idx]) for key in ['lr','lgb']};b17=Ensemble17(ms,ws,nb,rich,'add_both40');nnew=New19('cnb').fit([parsed17[i] for i in idx],yi[idx]);return Ensemble19(b17,{'cnb':nnew},'cnb10')\ndef base23(tr,va,tag):\n path=WORK/'artifacts/v19'/(tag+'.npz');meta=path.with_suffix('.json')\n if path.exists() and meta.exists():\n  info=json.loads(meta.read_text());assert info['sha']==SHA17 and info['protocol']==json.loads(json.dumps(PROTOCOL19))\n  with np.load(path) as q:\n   assert np.array_equal(q['tr'],tr) and np.array_equal(q['va'],va);return .9*q['base']+.1*q['cnb']\n old=fit_old23(tr);p=old.predict_frame(X.iloc[va]);del old;gc.collect();return p\n\ndef fit_fold23(tr,va,tag):\n assert not set(G17[tr])&set(G17[va]);assert not set(G17[tr])&set(G17[hold]);assert not set(va)&set(hold);path=OUT/(tag+'.npz');meta=path.with_suffix('.json')\n if path.exists() and meta.exists():\n  info=json.loads(meta.read_text());assert info['sha']==SHA17 and info['protocol']==json.loads(json.dumps(PROTOCOL23))\n  with np.load(path) as q:\n   assert np.array_equal(q['tr'],tr) and np.array_equal(q['va'],va);return {key:q[key] for key in q.files if key not in ['tr','va']}\n started=time.time();isconfirm=tag.startswith('confirmation');noise=None\n if isconfirm:\n  fold=int(tag.rsplit('fold',1)[1]);noise=rare_background(X.iloc[va],[events[i] for i in tr],23034+fold,count=35);old=fit_old23(tr);p={'base':old.predict_frame(X.iloc[va]),'base_noise35':old.predict_frame(noise)};del old;gc.collect()\n else:p={'base':base23(tr,va,tag)}\n print('V23 baseline ready',tag,round(time.time()-started,1),flush=True)\n for algo in ['ova','mlp']:\n  model=Model23(algo).fit([parsed17[i] for i in tr],yi[tr]);p[algo]=model.predict_proba([parsed17[i] for i in va])\n  if noise is not None:p[algo+'_noise35']=model.predict_frame(noise)\n  print('V23',tag,algo,'F1',round(f17(yi[va],p[algo]),5),'seconds',round(time.time()-started,1),flush=True);del model;gc.collect()\n for value in p.values():assert np.isfinite(value).all() and np.allclose(value.sum(1),1)\n np.savez_compressed(path,tr=tr,va=va,**p);meta.write_text(json.dumps({'sha':SHA17,'protocol':PROTOCOL23},indent=2));return p\n\ndef mix23(p,plan):return sum(w*p[key] for key,w in PLANS23[plan].items())\n", "PROTOCOL25={'plan':{'base':.6,'ova':.2,'mlp':.2}}\nclass Model25:\n\n    def __init__(self, models):\n        self.models = models\n        self.classes_ = labels.copy()\n        self.genes_ = list(genes)\n        self.training_data_sha256_ = SHA17\n\n    def predict_frame(self, frame):\n        canonical = canonical_frame25(frame.loc[:, self.genes_])\n        p = sum((w * self.models[k].predict_frame(canonical) for k, w in PROTOCOL25['plan'].items()))\n        assert np.isfinite(p).all() and np.allclose(p.sum(1), 1)\n        return p\n\ndef fit25(indices):\n    print('fresh base fit', len(indices), flush=True)\n    fitted = {'base': fit_old23(indices)}\n    for algo in ['ova', 'mlp']:\n        print('fresh', algo, 'fit', len(indices), flush=True)\n        fitted[algo] = Model23(algo).fit([parsed17[i] for i in indices], yi[indices])\n    return Model25(fitted)"], '26': ['import os,sys,re,json,pickle,cloudpickle,hashlib,platform,time,copy,warnings\nfrom pathlib import Path\nfrom collections import Counter\nimport numpy as np\nimport pandas as pd\nfrom scipy import sparse\nfrom scipy.special import softmax\nfrom sklearn.base import BaseEstimator,TransformerMixin,clone\nfrom sklearn.feature_extraction import DictVectorizer\nfrom sklearn.feature_selection import chi2,f_classif\nfrom sklearn.preprocessing import StandardScaler,MaxAbsScaler\nfrom sklearn.multiclass import OneVsRestClassifier\nfrom sklearn.model_selection import StratifiedGroupKFold\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.ensemble import ExtraTreesClassifier\nfrom sklearn.metrics import f1_score,accuracy_score,classification_report,confusion_matrix\nfrom lightgbm import LGBMClassifier\nfrom threadpoolctl import threadpool_limits\nfrom IPython.display import display,FileLink\nthreadpool_limits(2)\nSEED=42\nforbidden_attempts23=[]\ndef forbid_inference23(event,args):\n if event==\'open\' and args and isinstance(args[0],(str,bytes,os.PathLike)) and Path(os.fsdecode(args[0])).name in {\'test.csv\',\'sample_submission.csv\'}:\n  forbidden_attempts23.append(str(args[0]));raise RuntimeError(\'Inference data access is forbidden in v23\')\nsys.addaudithook(forbid_inference23)\nroots=[Path.cwd(),*Path.cwd().parents]\nREPO=next((p for p in roots if (p/\'data/train.csv\').is_file()),None)\nif REPO is None:raise FileNotFoundError(\'data/train.csv가 필요합니다.\')\nDATA=REPO/\'data\';WORK=REPO/\'models\'/\'혜진\' if (REPO/\'models\'/\'혜진\').is_dir() else REPO\nOUT=WORK/\'artifacts\'/\'v28\'/\'definitions_v26\';OUT.mkdir(parents=True,exist_ok=True)\n# Inference files and submission directory are intentionally not used.\nos.environ[\'MPLCONFIGDIR\']=str(OUT/\'.mplconfig\')\ntrain=pd.read_csv(DATA/\'train.csv\')\ngenes=[c for c in train if c not in [\'ID\',\'SUBCLASS\']]\nassert train.ID.is_unique\ny=train.SUBCLASS.to_numpy();labels=np.sort(np.unique(y));yi=np.searchsorted(labels,y);NC=len(labels)\n\ndef normalize_token25(token):\n return re.sub(r\'^(p\\.)?([ACDEFGHIKLMNPQRSTVWY])([1-9][0-9]*)X$\',lambda m:(m[1] or \'\')+m[2]+m[3]+\'*\',token)\ndef canonical_frame25(frame):\n def one(v):\n  if not isinstance(v,str) or \'X\' not in v:return v\n  return \' \'.join(normalize_token25(t) for t in v.split())\n return frame.map(one)\ndef legacy_frame25(frame,mixed=False):\n def one(v):\n  if not isinstance(v,str) or \'*\' not in v:return v\n  def conv(m):\n   token=m[0]\n   if mixed and int(hashlib.sha256(token.encode()).hexdigest()[:8],16)%2:return token\n   return token[:-1]+\'X\'\n  return re.sub(r\'(?<!\\S)(?:p\\.)?[ACDEFGHIKLMNPQRSTVWY][1-9][0-9]*\\*(?!\\S)\',conv,v)\n return frame.map(one)\n\nX25_original=train[genes].copy();X25_mixed=legacy_frame25(X25_original,mixed=True);X=canonical_frame25(X25_mixed);assert X.equals(X25_original);groups=pd.util.hash_pandas_object(X,index=False).to_numpy()\ndev,hold=next(StratifiedGroupKFold(5,shuffle=True,random_state=SEED).split(X,y,groups))\nfolds=list(StratifiedGroupKFold(3,shuffle=True,random_state=SEED+1).split(X.iloc[dev],y[dev],groups[dev]))\nassert not set(groups[dev]).intersection(groups[hold])\nfor a,b in folds:assert not set(groups[dev[a]]).intersection(groups[dev[b]])\npd.DataFrame({\'ID\':train.ID,\'split\':np.where(np.isin(np.arange(len(train)),hold),\'holdout\',\'development\'),\'group\':groups.astype(str)}).to_csv(OUT/\'split_manifest.csv\',index=False)\nprint(\'TRAIN ONLY\',train.shape,\'Dev/holdout\',len(dev),len(hold),\'OUT\',OUT,flush=True)\n\nfrom sklearn.neural_network import MLPClassifier\nfrom sklearn.feature_extraction.text import TfidfTransformer\nfrom sklearn.utils.class_weight import compute_sample_weight\nfrom xgboost import XGBClassifier\n\nfrom sklearn.pipeline import make_pipeline\nimport gc\nAA26="ACDEFGHIKLMNPQRSTVWY"\nTYPES26=["missense","silent","nonsense","frameshift","splice","indel","other"]\nCFG={"TMB_LOW":5,"TMB_HIGH":30}\ngene_pos={g:i for i,g in enumerate(genes)}\n', '# P43: HGVS 2016 principles; this is a conservative dataset parser, not a full HGVS validator.\nAA16=set(\'ACDEFGHIKLMNPQRSTVWY\');THREE16=dict(zip([\'Ala\',\'Arg\',\'Asn\',\'Asp\',\'Cys\',\'Gln\',\'Glu\',\'Gly\',\'His\',\'Ile\',\'Leu\',\'Lys\',\'Met\',\'Phe\',\'Pro\',\'Ser\',\'Thr\',\'Trp\',\'Tyr\',\'Val\',\'Ter\',\'Xaa\'],list(\'ARNDCQEGHILKMFPSTWYV\')+[\'*\',\'X\']))\n# Each event: key, kind, position, components(position,ref,alt,kind), flags, fs_stop_distance.\ndef parse_token16(raw):\n s=str(raw).strip();core=s[2:] if s.startswith(\'p.\') else s;uncertain=core.startswith(\'(\') and core.endswith(\')\');core=core[1:-1] if uncertain else core;core=normalize_token25(core)\n if re.match(\'^(?:\'+\'|\'.join(THREE16)+\')[0-9]+\',core):core=re.sub(\'|\'.join(THREE16),lambda m:THREE16[m[0]],core)\n core=normalize_token25(core)\n key=(\'uncertain|\' if uncertain else \'\')+core;flags=[\'uncertain\'] if uncertain else [];pos=None;parts=();dist=None;kind=\'unparsed\'\n def effect(a,b):\n  if a==\'*\' and b==\'*\':return \'stop_retained\'\n  if a==\'*\':return \'stop_lost\' if b in AA16 else \'ambiguous_residue\'\n  if b==\'*\':return \'stop_gained\' if a in AA16 else \'ambiguous_residue\'\n  if a not in AA16 or b not in AA16:return \'ambiguous_residue\'\n  return \'synonymous\' if a==b else \'missense\'\n m=re.fullmatch(r\'([A-Z*])([1-9][0-9]*)([A-Z*])\',core)\n if m:\n  a,p,b=m.groups();pos=int(p);kind=effect(a,b);parts=((pos,a,b,kind),)\n  if \'X\' in [a,b]:flags.append(\'ambiguous_X\')\n else:\n  m=re.fullmatch(r\'([1-9][0-9]*)_([1-9][0-9]*)([A-Z*]+)>([A-Z*]+)\',core)\n  if m:\n   start,end,ref,alt=m.groups();start=int(start);end=int(end);pos=start;span=end-start+1\n   if span>0 and span==len(ref)==len(alt) and all(a in AA16 or a==\'*\' for a in ref+alt):\n    parts=tuple((start+j,a,b,effect(a,b)) for j,(a,b) in enumerate(zip(ref,alt)));kind=\'synonymous_block\' if all(a==b for a,b in zip(ref,alt)) else \'block_change\';flags.append(\'resolved_equal_length_block\')\n   else:kind=\'complex_range\';flags.append(\'unresolved_range\')\n  elif \'fs\' in core:\n   kind=\'frameshift\';m=re.match(r\'([A-Z*]+)([1-9][0-9]*)\',core);pos=int(m[2]) if m else None;m=re.search(r\'fs(?:\\*|Ter)([1-9][0-9]*)$\',core);dist=int(m[1]) if m else None\n  elif \'delins\' in core:kind=\'delins\'\n  elif \'del\' in core:kind=\'deletion\'\n  elif \'ins\' in core:kind=\'insertion\'\n  elif \'dup\' in core:kind=\'duplication\'\n  elif \'ext\' in core:kind=\'extension\'\n  elif core==\'=\':kind=\'no_change_unspecified\'\n  elif re.fullmatch(r\'[A-Z*]+[1-9][0-9]*\',core):kind=\'incomplete\';flags.append(\'missing_alternate\')\n  if pos is None:\n   # Protein coordinate only: do not mistake fs termination distance for the affected residue.\n   m=re.match(r\'(?:[A-Z*]+)?([1-9][0-9]*)\',core);pos=int(m[1]) if m else None\n return (key,kind,pos,parts,tuple(flags),dist)\n\ndef parse_rows16(frame):\n cols=list(frame.columns);cache={};rows=[]\n for row in frame.itertuples(index=False,name=None):\n  mutations=[];missing=[];raw_n=0;duplicates=0\n  for gene,value in zip(cols,row):\n   if value is None or pd.isna(value) or (isinstance(value,str) and not value.strip()):missing.append(gene);continue\n   if str(value).strip()==\'WT\':continue\n   tokens=str(value).split();raw_n+=len(tokens);tokens=[normalize_token25(t) for t in tokens];unique=list(dict.fromkeys(tokens));duplicates+=len(tokens)-len(unique);events=[]\n   for token in unique:\n    if token not in cache:cache[token]=parse_token16(token)\n    events.append(cache[token])\n   mutations.append((gene,tuple(events)))\n  rows.append({\'mutations\':tuple(mutations),\'missing\':tuple(missing),\'raw_tokens\':raw_n,\'duplicate_tokens\':duplicates})\n return rows\n\nFIXTURES16=[(\'R132H\',\'missense\',132),(\'R132R\',\'synonymous\',132),(\'R213*\',\'stop_gained\',213),(\'*363*\',\'stop_retained\',363),(\'*213Q\',\'stop_lost\',213),(\'K16Rfs*23\',\'frameshift\',16),(\'1_2FS>FS\',\'synonymous_block\',1),(\'483_484MP>IA\',\'block_change\',483),(\'10_12AAA>DD\',\'complex_range\',10),(\'490del\',\'deletion\',490),(\'p.(Arg123His)\',\'missense\',123),(\'R123X\',\'stop_gained\',123),(\'A391\',\'incomplete\',391),(\'FS\',\'unparsed\',None),(\'G12delinsAA\',\'delins\',12)]\nfor token,expected_kind16,expected_pos16 in FIXTURES16:\n result=parse_token16(token);assert result[1]==expected_kind16 and result[2]==expected_pos16,(token,result)\nassert parse_token16(\'K16Rfs*23\')[5]==23\nassert parse_token16(\'1_2FS>FS\')[3]==((1,\'F\',\'F\',\'synonymous\'),(2,\'S\',\'S\',\'synonymous\'))\nassert \'uncertain\' in parse_token16(\'p.(Arg123His)\')[4]\nassert parse_rows16(pd.DataFrame({\'G\':[\'WT\',None,\'R1H R1H\']}))[1][\'missing\']==(\'G\',)\nassert parse_rows16(pd.DataFrame({\'G\':[\'WT\',None,\'R1H R1H\']}))[2][\'duplicate_tokens\']==1\nprint(\'P43 annotation safety fixtures passed:\',len(FIXTURES16),flush=True)\n# P25-P40: small, explicit gene subsets in the publications, not whole pathways or diagnostic rules.\nPANELS16={\n \'P25_breast\':(\'TP53\',\'PIK3CA\',\'GATA3\',\'MAP3K1\'),\n \'P26_colorectal\':(\'APC\',\'TP53\',\'SMAD4\',\'PIK3CA\',\'KRAS\',\'ARID1A\',\'SOX9\',\'FAM123B\',\'POLE\',\'MLH1\'),\n \'P27_glioma\':(\'IDH1\',\'IDH2\',\'TP53\',\'ATRX\',\'CIC\',\'FUBP1\',\'NOTCH1\'),\n \'P29_gbm\':(\'EGFR\',\'PDGFRA\'),\n \'P30_renal\':(\'VHL\',\'PBRM1\',\'SETD2\',\'ARID1A\',\'SMARCA4\'),\n \'P31_papillary_renal\':(\'MET\',\'SETD2\',\'FH\'),\n \'P32_ovarian\':(\'TP53\',\'NF1\',\'BRCA1\',\'BRCA2\',\'RB1\',\'CDK12\'),\n \'P33_uterine\':(\'TP53\',\'PTEN\',\'CTNNB1\',\'PIK3CA\',\'ARID1A\',\'KRAS\',\'ARID5B\',\'POLE\'),\n \'P34_thyroid\':(\'BRAF\',\'EIF1AX\',\'PPM1D\',\'CHEK2\'),\n \'P35_prostate\':(\'SPOP\',\'FOXA1\',\'IDH1\'),\n \'P36_lung_ad\':(\'RIT1\',\'MGA\',\'EGFR\',\'RBM10\',\'NF1\',\'MET\',\'ERBB2\'),\n \'P37_lung_scc\':(\'TP53\',\'HLA-A\',\'NFE2L2\',\'KEAP1\',\'CDKN2A\',\'RB1\'),\n \'P38_melanoma\':(\'BRAF\',\'NRAS\',\'HRAS\',\'KRAS\',\'NF1\',\'KIT\'),\n \'P39_gastric\':(\'PIK3CA\',\'RHOA\'),\n \'P40_headneck\':(\'PIK3CA\',\'TRAF3\',\'TP53\',\'CDKN2A\',\'HRAS\',\'CASP8\',\'NOTCH1\',\'NSD1\',\'AJUBA\',\'FAT1\',\'NFE2L2\')}\nprint(\'Literature panel subsets:\',len(PANELS16),\'(external biological prior; no external patient data)\',flush=True)\n\nassert parse_token16("p.Arg123X")==parse_token16("R123*")\n', 'GENES16=list(genes)\nparsed17=parse_rows16(X)\n\nparent17=np.arange(len(train))\ndef root17(i):\n while parent17[i]!=i:parent17[i]=parent17[parent17[i]];i=parent17[i]\n return i\ndef join17(a,b):parent17[root17(b)]=root17(a)\ncanonical17=[hashlib.sha256(repr((tuple(sorted((g,tuple(sorted(set(e[0] for e in ev)))) for g,ev in row[\'mutations\'])),tuple(sorted(row[\'missing\'])))).encode()).hexdigest() for row in parsed17]\nfor identifiers in [groups,canonical17]:\n seen17={}\n for i,key in enumerate(identifiers):\n  if key in seen17:join17(seen17[key],i)\n  else:seen17[key]=i\nG17=np.array([root17(i) for i in range(len(train))]);overlap17=set(G17[dev])&set(G17[hold]);D17=dev[~np.isin(G17[dev],list(overlap17))]\nassert not set(G17[D17])&set(G17[hold])\n\nSHA17=hashlib.sha256((DATA/"train.csv").read_bytes()).hexdigest()\nprint("Development groups",len(D17),flush=True)\n', '# Adapted from supplied model_민정_v6.ipynb cells10/14. No teammate data or outputs executed.\nPATHWAYS = {\'p53\': [\'TP53\', \'MDM2\', \'MDM4\', \'CDKN2A\', \'ATM\', \'CHEK1\', \'CHEK2\'], \'RTK_RAS\': [\'EGFR\', \'ERBB2\', \'ERBB3\', \'ERBB4\', \'FGFR1\', \'FGFR2\', \'FGFR3\', \'FGFR4\', \'MET\', \'ALK\', \'RET\', \'KIT\', \'PDGFRA\', \'KRAS\', \'NRAS\', \'HRAS\', \'BRAF\', \'RAF1\', \'NF1\', \'MAP2K1\', \'MAP2K2\', \'SOS1\', \'PTPN11\'], \'PI3K\': [\'PIK3CA\', \'PIK3CB\', \'PIK3R1\', \'PTEN\', \'AKT1\', \'AKT2\', \'AKT3\', \'MTOR\', \'TSC1\', \'TSC2\', \'STK11\', \'RICTOR\', \'INPP4B\'], \'Wnt\': [\'APC\', \'CTNNB1\', \'AXIN1\', \'AXIN2\', \'RNF43\', \'ZNRF3\', \'TCF7L2\', \'GSK3B\', \'AMER1\'], \'CellCycle\': [\'RB1\', \'CDK4\', \'CDK6\', \'CCND1\', \'CCND2\', \'CCND3\', \'CCNE1\', \'CDKN1A\', \'CDKN1B\', \'CDKN2B\', \'E2F1\', \'FBXW7\'], \'Notch\': [\'NOTCH1\', \'NOTCH2\', \'NOTCH3\', \'NOTCH4\', \'FBXW7\', \'SPEN\'], \'Hippo\': [\'NF2\', \'LATS1\', \'LATS2\', \'SAV1\', \'YAP1\', \'WWTR1\', \'TEAD1\'], \'TGFb\': [\'TGFBR1\', \'TGFBR2\', \'SMAD2\', \'SMAD3\', \'SMAD4\', \'ACVR1B\', \'ACVR2A\'], \'NRF2\': [\'NFE2L2\', \'KEAP1\', \'CUL3\'], \'SWI_SNF\': [\'ARID1A\', \'ARID1B\', \'ARID2\', \'SMARCA4\', \'SMARCB1\', \'PBRM1\', \'SMARCA2\'], \'Chromatin\': [\'KMT2A\', \'KMT2B\', \'KMT2C\', \'KMT2D\', \'EZH2\', \'SETD2\', \'KDM6A\', \'KDM5C\', \'CREBBP\', \'EP300\', \'BAP1\', \'ASXL1\', \'DNMT3A\', \'IDH1\', \'IDH2\', \'ATRX\'], \'DDR_MMR\': [\'BRCA1\', \'BRCA2\', \'PALB2\', \'ATR\', \'POLE\', \'POLD1\', \'MLH1\', \'MSH2\', \'MSH6\', \'PMS2\', \'RAD51\', \'RAD50\', \'MRE11\', \'FANCA\', \'FANCD2\', \'ERCC2\'], \'Splicing\': [\'SF3B1\', \'SRSF2\', \'U2AF1\', \'RBM10\'], \'LongGenes\': [\'TTN\', \'MUC16\', \'MUC4\', \'SYNE1\', \'OBSCN\', \'RYR2\', \'RYR1\', \'CSMD1\', \'CSMD3\', \'LRP1B\', \'USH2A\', \'FLG\', \'PCLO\']}\n\ndef combo_features(ex):\n    mut = ex["mut"]; n = mut.shape[0]\n    n_genes = mut.sum(1)\n    low, high = n_genes <= CFG["TMB_LOW"], n_genes >= CFG["TMB_HIGH"]\n\n    def G(name):   # 유전자 변이 여부 (없는 유전자는 0 벡터 — 엉뚱한 컬럼 안 가져옴)\n        j = gene_pos.get(name)\n        return mut[:, j].copy() if j is not None else np.zeros(n, bool)\n\n    tok_rows = {}\n    for r, nm in zip(ex["tok_rows"], ex["tok_names"]):\n        tok_rows.setdefault(nm, []).append(r)\n    def T(name):   # 핫스팟/절단형 토큰 보유 여부 (MIN_HOTSPOT_FREQ와 무관하게 직접 계산)\n        v = np.zeros(n, bool)\n        if name in tok_rows: v[tok_rows[name]] = True\n        return v\n\n    f = {}\n    # COAD\n    f["COAD_APCtr_TP53"]  = T("APC|trunc") & G("TP53")\n    f["COAD_APCtr_KRAS"]  = T("APC|trunc") & (G("KRAS") | G("FBXW7"))\n    f["COAD_score"]       = T("APC|trunc").astype(int) + G("TP53") + G("KRAS") + G("FBXW7") + G("PIK3CA")\n    # KIRC / KIPAN\n    f["KIRC_VHL_PBRM1"]   = G("VHL") & G("PBRM1")\n    f["KIRC_3p_score"]    = G("VHL").astype(int) + G("PBRM1") + G("SETD2") + G("BAP1")\n    # UCEC\n    f["UCEC_PTEN_CTNNB1"] = G("PTEN") & G("CTNNB1")\n    f["UCEC_PTEN_PIK3CA"] = G("PTEN") & G("PIK3CA")\n    f["UCEC_highTMB_PTEN"]= high & G("PTEN")\n    # PAAD\n    f["PAAD_score"]       = T("KRAS|G12").astype(int) + G("TP53") + T("CDKN2A|trunc") + G("SMAD4")\n    # LGG vs GBMLGG\n    f["LGG_IDH1_TP53_ATRX"] = T("IDH1|R132") & G("TP53") & G("ATRX")\n    f["LGG_IDH1_clean"]   = T("IDH1|R132") & ~(G("PTEN") | G("EGFR"))\n    f["GBM_PTEN_EGFR"]    = G("PTEN") & G("EGFR")\n    f["GBM_noIDH1_signal"]= (G("PTEN") | G("EGFR")) & ~G("IDH1")\n    # BRAF V600 + TMB (THCA vs SKCM vs COAD)\n    f["THCA_BRAF_lowTMB"] = T("BRAF|V600") & low\n    f["SKCM_BRAF_highTMB"]= T("BRAF|V600") & high\n    f["SKCM_NRAS_highTMB"]= T("NRAS|Q61") & high\n    f["COAD_BRAF_APC"]    = T("BRAF|V600") & T("APC|trunc")\n    # 폐암 / 위식도 / 간 / 난소 / 백혈병\n    f["LUAD_EGFR_noTP53"] = G("EGFR") & ~G("TP53")\n    f["LUSC_TP53_noEGFR_highTMB"] = G("TP53") & ~G("EGFR") & high\n    f["STES_highTMB_TP53_noAPC"]  = high & G("TP53") & ~T("APC|trunc")\n    f["LIHC_CTNNB1_noAPC"]= G("CTNNB1") & ~G("APC")\n    f["OV_TP53_lowTMB"]   = G("TP53") & (n_genes <= 20)\n    f["LAML_NPM1_lowTMB"] = G("NPM1") & low\n\n    names = list(f.keys())\n    return np.c_[[np.asarray(f[k], np.float32) for k in names]].T, names', "TRUNC26={'stop_gained','frameshift'}\nINDEL26={'deletion','insertion','delins','duplication'}\nclass Features26:\n def __init__(self,combos=True):self.combos=combos\n def extract(self,rows):\n  gp={g:i for i,g in enumerate(genes)};mut=np.zeros((len(rows),len(genes)),bool);tc=np.zeros((len(rows),7),np.float32);ac=np.zeros((len(rows),400),np.float32);rr=[];tt=[]\n  for r,row in enumerate(rows):\n   for g,events26 in row['mutations']:\n    mut[r,gp[g]]=True\n    for key,kind,pos,parts,flags,dist in events26:\n     typ='nonsense' if kind=='stop_gained' else ('silent' if kind in {'synonymous','synonymous_block'} else ('indel' if kind in INDEL26 else (kind if kind in {'missense','frameshift'} else 'other')))\n     tc[r,TYPES26.index(typ)]+=1\n     if kind=='missense' and parts:\n      _,a,b,_=parts[0];rr.append(r);tt.append(f'{g}|{a}{pos}')\n      if a in AA26 and b in AA26:ac[r,AA26.index(a)*20+AA26.index(b)]+=1\n     elif kind in TRUNC26:rr.append(r);tt.append(f'{g}|trunc')\n     elif kind in INDEL26:\n      rr.extend([r,r]);tt.extend([f'{g}|indel',f'{g}|{pos}_indel'])\n  return {'mut':mut,'type_cnt':tc,'aa_cnt':ac,'tok_rows':np.asarray(rr,dtype=int),'tok_names':tt}\n def fit(self,rows):\n  ex=self.extract(rows);self.keep=np.flatnonzero(ex['mut'].sum(0)>=5);cnt=Counter(nm for _,nm in set(zip(ex['tok_rows'],ex['tok_names'])));self.tokens=sorted(k for k,v in cnt.items() if v>=5);self.tidx={k:i for i,k in enumerate(self.tokens)};self.aakeep=np.flatnonzero(ex['aa_cnt'].sum(0)>=20)\n  self.names=self.compose(ex)[1];return self\n def compose(self,ex):\n  mut=ex['mut'];tc=ex['type_cnt'];n=len(mut);ng=mut.sum(1);nev=tc.sum(1);blocks=[];names=[]\n  def add(a,nm):blocks.append(np.asarray(a,np.float32).reshape(n,-1));names.extend(nm)\n  add(mut[:,self.keep],[f'g_{genes[j]}' for j in self.keep]);add(np.c_[ng,np.log1p(ng),nev],['n_mut_genes','log_n_mut_genes','n_events']);add(tc,['cnt_'+s for s in TYPES26]);trunc=tc[:,[TYPES26.index('nonsense'),TYPES26.index('frameshift')]].sum(1)\n  add(np.c_[tc[:,1]/(nev+1),tc[:,0]/(nev+1),trunc/(nev+1)],['ratio_silent','ratio_missense','ratio_trunc'])\n  h=np.zeros((n,len(self.tokens)),np.float32)\n  for r,t in zip(ex['tok_rows'],ex['tok_names']):\n   j=self.tidx.get(t)\n   if j is not None:h[r,j]=1\n  add(h,['hs_'+s for s in self.tokens])\n  if self.combos:\n   c,cn=combo_features(ex);add(c,['cb_'+s for s in cn])\n  if len(self.aakeep):add(ex['aa_cnt'][:,self.aakeep]/(tc[:,0]+1)[:,None],[f'aa_{AA26[j//20]}>{AA26[j%20]}' for j in self.aakeep])\n  add(np.column_stack([mut[:,[gene_pos[g] for g in gs if g in gene_pos]].sum(1) for gs in PATHWAYS.values()]),['pw_'+k for k in PATHWAYS])\n  return np.hstack(blocks),names\n def transform(self,rows):return self.compose(self.extract(rows))[0]\n\ndef new_xgb26(rounds,early=False):\n return XGBClassifier(objective='multi:softprob',n_estimators=rounds,learning_rate=.08,max_depth=5,subsample=.8,colsample_bytree=.3,tree_method='hist',eval_metric='mlogloss',random_state=26026,n_jobs=2,**({'early_stopping_rounds':50} if early else {}))\nclass Model26:\n def __init__(self,combos=True):self.combos=combos;self.classes_=labels.copy();self.genes_=list(genes)\n def fit(self,indices,rounds=None,seed=26026):\n  if rounds is None:\n   ia,ib=next(StratifiedGroupKFold(5,shuffle=True,random_state=seed).split(np.zeros(len(indices)),yi[indices],G17[indices]));it,iv=indices[ia],indices[ib];assert not set(G17[it])&set(G17[iv]);innerpre=Features26(self.combos).fit([parsed17[i] for i in it]);a=innerpre.transform([parsed17[i] for i in it]);b=innerpre.transform([parsed17[i] for i in iv]);inner=new_xgb26(1200,True)\n   print('V26 inner early stopping',self.combos,len(it),len(iv),len(innerpre.names),flush=True);inner.fit(a,yi[it],sample_weight=compute_sample_weight('balanced',yi[it]),eval_set=[(b,yi[iv])],verbose=False);rounds=int(inner.best_iteration)+1;del innerpre,inner,a,b;gc.collect()\n  self.rounds_=int(rounds);self.pre=Features26(self.combos).fit([parsed17[i] for i in indices]);a=self.pre.transform([parsed17[i] for i in indices]);self.xgb=new_xgb26(self.rounds_)\n  print('V26 outer/full refit',self.combos,len(indices),len(self.pre.names),self.rounds_,flush=True);self.xgb.fit(a,yi[indices],sample_weight=compute_sample_weight('balanced',yi[indices]),verbose=False)\n  self.lr=make_pipeline(MaxAbsScaler(),LogisticRegression(C=.1,class_weight='balanced',max_iter=1000));self.lr.fit(a,yi[indices]);return self\n def predict_parts(self,rows):\n  a=self.pre.transform(rows);return self.xgb.predict_proba(a),self.lr.predict_proba(a)\n def predict_rows(self,rows):\n  a,b=self.predict_parts(rows);p=.5*a+.5*b;assert p.shape==(len(rows),len(self.classes_)) and np.isfinite(p).all() and np.allclose(p.sum(1),1);return p\n def predict_frame(self,frame):return self.predict_rows(parse_rows16(canonical_frame25(frame.loc[:,self.genes_])))\n"], '27': ['import os,sys,re,json,pickle,cloudpickle,hashlib,platform,time,copy,warnings\nfrom pathlib import Path\nfrom collections import Counter\nimport numpy as np\nimport pandas as pd\nfrom scipy import sparse\nfrom scipy.special import softmax\nfrom sklearn.base import BaseEstimator,TransformerMixin,clone\nfrom sklearn.feature_extraction import DictVectorizer\nfrom sklearn.feature_selection import chi2,f_classif\nfrom sklearn.preprocessing import StandardScaler,MaxAbsScaler\nfrom sklearn.multiclass import OneVsRestClassifier\nfrom sklearn.model_selection import StratifiedGroupKFold\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.ensemble import ExtraTreesClassifier\nfrom sklearn.metrics import f1_score,accuracy_score,classification_report,confusion_matrix\nfrom lightgbm import LGBMClassifier\nfrom threadpoolctl import threadpool_limits\nfrom IPython.display import display,FileLink\nthreadpool_limits(2)\nSEED=42\nforbidden_attempts23=[]\ndef forbid_inference23(event,args):\n if event==\'open\' and args and isinstance(args[0],(str,bytes,os.PathLike)) and Path(os.fsdecode(args[0])).name in {\'test.csv\',\'sample_submission.csv\'}:\n  forbidden_attempts23.append(str(args[0]));raise RuntimeError(\'Inference data access is forbidden in v23\')\nsys.addaudithook(forbid_inference23)\nroots=[Path.cwd(),*Path.cwd().parents]\nREPO=next((p for p in roots if (p/\'data/train.csv\').is_file()),None)\nif REPO is None:raise FileNotFoundError(\'data/train.csv가 필요합니다.\')\nDATA=REPO/\'data\';WORK=REPO/\'models\'/\'혜진\' if (REPO/\'models\'/\'혜진\').is_dir() else REPO\nOUT=WORK/\'artifacts\'/\'v28\'/\'definitions_v27\';OUT.mkdir(parents=True,exist_ok=True)\n# Inference files and submission directory are intentionally not used.\nos.environ[\'MPLCONFIGDIR\']=str(OUT/\'.mplconfig\')\ntrain=pd.read_csv(DATA/\'train.csv\')\ngenes=[c for c in train if c not in [\'ID\',\'SUBCLASS\']]\nassert train.ID.is_unique\ny=train.SUBCLASS.to_numpy();labels=np.sort(np.unique(y));yi=np.searchsorted(labels,y);NC=len(labels)\n\ndef normalize_token25(token):\n return re.sub(r\'^(p\\.)?([ACDEFGHIKLMNPQRSTVWY])([1-9][0-9]*)X$\',lambda m:(m[1] or \'\')+m[2]+m[3]+\'*\',token)\ndef canonical_frame25(frame):\n def one(v):\n  if not isinstance(v,str) or \'X\' not in v:return v\n  return \' \'.join(normalize_token25(t) for t in v.split())\n return frame.map(one)\ndef legacy_frame25(frame,mixed=False):\n def one(v):\n  if not isinstance(v,str) or \'*\' not in v:return v\n  def conv(m):\n   token=m[0]\n   if mixed and int(hashlib.sha256(token.encode()).hexdigest()[:8],16)%2:return token\n   return token[:-1]+\'X\'\n  return re.sub(r\'(?<!\\S)(?:p\\.)?[ACDEFGHIKLMNPQRSTVWY][1-9][0-9]*\\*(?!\\S)\',conv,v)\n return frame.map(one)\n\nX25_original=train[genes].copy();X25_mixed=legacy_frame25(X25_original,mixed=True);X=canonical_frame25(X25_mixed);assert X.equals(X25_original);groups=pd.util.hash_pandas_object(X,index=False).to_numpy()\ndev,hold=next(StratifiedGroupKFold(5,shuffle=True,random_state=SEED).split(X,y,groups))\nfolds=list(StratifiedGroupKFold(3,shuffle=True,random_state=SEED+1).split(X.iloc[dev],y[dev],groups[dev]))\nassert not set(groups[dev]).intersection(groups[hold])\nfor a,b in folds:assert not set(groups[dev[a]]).intersection(groups[dev[b]])\npd.DataFrame({\'ID\':train.ID,\'split\':np.where(np.isin(np.arange(len(train)),hold),\'holdout\',\'development\'),\'group\':groups.astype(str)}).to_csv(OUT/\'split_manifest.csv\',index=False)\nprint(\'TRAIN ONLY\',train.shape,\'Dev/holdout\',len(dev),len(hold),\'OUT\',OUT,flush=True)\n\nfrom sklearn.neural_network import MLPClassifier\nfrom sklearn.feature_extraction.text import TfidfTransformer\nfrom sklearn.utils.class_weight import compute_sample_weight\nfrom xgboost import XGBClassifier\n\nfrom sklearn.pipeline import make_pipeline\nimport gc\nAA26="ACDEFGHIKLMNPQRSTVWY"\nTYPES26=["missense","silent","nonsense","frameshift","splice","indel","other"]\nCFG={"TMB_LOW":5,"TMB_HIGH":30}\ngene_pos={g:i for i,g in enumerate(genes)}\n\nfrom lightgbm import early_stopping\n', '# P43: HGVS 2016 principles; this is a conservative dataset parser, not a full HGVS validator.\nAA16=set(\'ACDEFGHIKLMNPQRSTVWY\');THREE16=dict(zip([\'Ala\',\'Arg\',\'Asn\',\'Asp\',\'Cys\',\'Gln\',\'Glu\',\'Gly\',\'His\',\'Ile\',\'Leu\',\'Lys\',\'Met\',\'Phe\',\'Pro\',\'Ser\',\'Thr\',\'Trp\',\'Tyr\',\'Val\',\'Ter\',\'Xaa\'],list(\'ARNDCQEGHILKMFPSTWYV\')+[\'*\',\'X\']))\n# Each event: key, kind, position, components(position,ref,alt,kind), flags, fs_stop_distance.\ndef parse_token16(raw):\n s=str(raw).strip();core=s[2:] if s.startswith(\'p.\') else s;uncertain=core.startswith(\'(\') and core.endswith(\')\');core=core[1:-1] if uncertain else core;core=normalize_token25(core)\n if re.match(\'^(?:\'+\'|\'.join(THREE16)+\')[0-9]+\',core):core=re.sub(\'|\'.join(THREE16),lambda m:THREE16[m[0]],core)\n core=normalize_token25(core)\n key=(\'uncertain|\' if uncertain else \'\')+core;flags=[\'uncertain\'] if uncertain else [];pos=None;parts=();dist=None;kind=\'unparsed\'\n def effect(a,b):\n  if a==\'*\' and b==\'*\':return \'stop_retained\'\n  if a==\'*\':return \'stop_lost\' if b in AA16 else \'ambiguous_residue\'\n  if b==\'*\':return \'stop_gained\' if a in AA16 else \'ambiguous_residue\'\n  if a not in AA16 or b not in AA16:return \'ambiguous_residue\'\n  return \'synonymous\' if a==b else \'missense\'\n m=re.fullmatch(r\'([A-Z*])([1-9][0-9]*)([A-Z*])\',core)\n if m:\n  a,p,b=m.groups();pos=int(p);kind=effect(a,b);parts=((pos,a,b,kind),)\n  if \'X\' in [a,b]:flags.append(\'ambiguous_X\')\n else:\n  m=re.fullmatch(r\'([1-9][0-9]*)_([1-9][0-9]*)([A-Z*]+)>([A-Z*]+)\',core)\n  if m:\n   start,end,ref,alt=m.groups();start=int(start);end=int(end);pos=start;span=end-start+1\n   if span>0 and span==len(ref)==len(alt) and all(a in AA16 or a==\'*\' for a in ref+alt):\n    parts=tuple((start+j,a,b,effect(a,b)) for j,(a,b) in enumerate(zip(ref,alt)));kind=\'synonymous_block\' if all(a==b for a,b in zip(ref,alt)) else \'block_change\';flags.append(\'resolved_equal_length_block\')\n   else:kind=\'complex_range\';flags.append(\'unresolved_range\')\n  elif \'fs\' in core:\n   kind=\'frameshift\';m=re.match(r\'([A-Z*]+)([1-9][0-9]*)\',core);pos=int(m[2]) if m else None;m=re.search(r\'fs(?:\\*|Ter)([1-9][0-9]*)$\',core);dist=int(m[1]) if m else None\n  elif \'delins\' in core:kind=\'delins\'\n  elif \'del\' in core:kind=\'deletion\'\n  elif \'ins\' in core:kind=\'insertion\'\n  elif \'dup\' in core:kind=\'duplication\'\n  elif \'ext\' in core:kind=\'extension\'\n  elif core==\'=\':kind=\'no_change_unspecified\'\n  elif re.fullmatch(r\'[A-Z*]+[1-9][0-9]*\',core):kind=\'incomplete\';flags.append(\'missing_alternate\')\n  if pos is None:\n   # Protein coordinate only: do not mistake fs termination distance for the affected residue.\n   m=re.match(r\'(?:[A-Z*]+)?([1-9][0-9]*)\',core);pos=int(m[1]) if m else None\n return (key,kind,pos,parts,tuple(flags),dist)\n\ndef parse_rows16(frame):\n cols=list(frame.columns);cache={};rows=[]\n for row in frame.itertuples(index=False,name=None):\n  mutations=[];missing=[];raw_n=0;duplicates=0\n  for gene,value in zip(cols,row):\n   if value is None or pd.isna(value) or (isinstance(value,str) and not value.strip()):missing.append(gene);continue\n   if str(value).strip()==\'WT\':continue\n   tokens=str(value).split();raw_n+=len(tokens);tokens=[normalize_token25(t) for t in tokens];unique=list(dict.fromkeys(tokens));duplicates+=len(tokens)-len(unique);events=[]\n   for token in unique:\n    if token not in cache:cache[token]=parse_token16(token)\n    events.append(cache[token])\n   mutations.append((gene,tuple(events)))\n  rows.append({\'mutations\':tuple(mutations),\'missing\':tuple(missing),\'raw_tokens\':raw_n,\'duplicate_tokens\':duplicates})\n return rows\n\nFIXTURES16=[(\'R132H\',\'missense\',132),(\'R132R\',\'synonymous\',132),(\'R213*\',\'stop_gained\',213),(\'*363*\',\'stop_retained\',363),(\'*213Q\',\'stop_lost\',213),(\'K16Rfs*23\',\'frameshift\',16),(\'1_2FS>FS\',\'synonymous_block\',1),(\'483_484MP>IA\',\'block_change\',483),(\'10_12AAA>DD\',\'complex_range\',10),(\'490del\',\'deletion\',490),(\'p.(Arg123His)\',\'missense\',123),(\'R123X\',\'stop_gained\',123),(\'A391\',\'incomplete\',391),(\'FS\',\'unparsed\',None),(\'G12delinsAA\',\'delins\',12)]\nfor token,expected_kind16,expected_pos16 in FIXTURES16:\n result=parse_token16(token);assert result[1]==expected_kind16 and result[2]==expected_pos16,(token,result)\nassert parse_token16(\'K16Rfs*23\')[5]==23\nassert parse_token16(\'1_2FS>FS\')[3]==((1,\'F\',\'F\',\'synonymous\'),(2,\'S\',\'S\',\'synonymous\'))\nassert \'uncertain\' in parse_token16(\'p.(Arg123His)\')[4]\nassert parse_rows16(pd.DataFrame({\'G\':[\'WT\',None,\'R1H R1H\']}))[1][\'missing\']==(\'G\',)\nassert parse_rows16(pd.DataFrame({\'G\':[\'WT\',None,\'R1H R1H\']}))[2][\'duplicate_tokens\']==1\nprint(\'P43 annotation safety fixtures passed:\',len(FIXTURES16),flush=True)\n# P25-P40: small, explicit gene subsets in the publications, not whole pathways or diagnostic rules.\nPANELS16={\n \'P25_breast\':(\'TP53\',\'PIK3CA\',\'GATA3\',\'MAP3K1\'),\n \'P26_colorectal\':(\'APC\',\'TP53\',\'SMAD4\',\'PIK3CA\',\'KRAS\',\'ARID1A\',\'SOX9\',\'FAM123B\',\'POLE\',\'MLH1\'),\n \'P27_glioma\':(\'IDH1\',\'IDH2\',\'TP53\',\'ATRX\',\'CIC\',\'FUBP1\',\'NOTCH1\'),\n \'P29_gbm\':(\'EGFR\',\'PDGFRA\'),\n \'P30_renal\':(\'VHL\',\'PBRM1\',\'SETD2\',\'ARID1A\',\'SMARCA4\'),\n \'P31_papillary_renal\':(\'MET\',\'SETD2\',\'FH\'),\n \'P32_ovarian\':(\'TP53\',\'NF1\',\'BRCA1\',\'BRCA2\',\'RB1\',\'CDK12\'),\n \'P33_uterine\':(\'TP53\',\'PTEN\',\'CTNNB1\',\'PIK3CA\',\'ARID1A\',\'KRAS\',\'ARID5B\',\'POLE\'),\n \'P34_thyroid\':(\'BRAF\',\'EIF1AX\',\'PPM1D\',\'CHEK2\'),\n \'P35_prostate\':(\'SPOP\',\'FOXA1\',\'IDH1\'),\n \'P36_lung_ad\':(\'RIT1\',\'MGA\',\'EGFR\',\'RBM10\',\'NF1\',\'MET\',\'ERBB2\'),\n \'P37_lung_scc\':(\'TP53\',\'HLA-A\',\'NFE2L2\',\'KEAP1\',\'CDKN2A\',\'RB1\'),\n \'P38_melanoma\':(\'BRAF\',\'NRAS\',\'HRAS\',\'KRAS\',\'NF1\',\'KIT\'),\n \'P39_gastric\':(\'PIK3CA\',\'RHOA\'),\n \'P40_headneck\':(\'PIK3CA\',\'TRAF3\',\'TP53\',\'CDKN2A\',\'HRAS\',\'CASP8\',\'NOTCH1\',\'NSD1\',\'AJUBA\',\'FAT1\',\'NFE2L2\')}\nprint(\'Literature panel subsets:\',len(PANELS16),\'(external biological prior; no external patient data)\',flush=True)\n\nassert parse_token16("p.Arg123X")==parse_token16("R123*")\n', 'GENES16=list(genes)\nparsed17=parse_rows16(X)\n\nparent17=np.arange(len(train))\ndef root17(i):\n while parent17[i]!=i:parent17[i]=parent17[parent17[i]];i=parent17[i]\n return i\ndef join17(a,b):parent17[root17(b)]=root17(a)\ncanonical17=[hashlib.sha256(repr((tuple(sorted((g,tuple(sorted(set(e[0] for e in ev)))) for g,ev in row[\'mutations\'])),tuple(sorted(row[\'missing\'])))).encode()).hexdigest() for row in parsed17]\nfor identifiers in [groups,canonical17]:\n seen17={}\n for i,key in enumerate(identifiers):\n  if key in seen17:join17(seen17[key],i)\n  else:seen17[key]=i\nG17=np.array([root17(i) for i in range(len(train))]);overlap17=set(G17[dev])&set(G17[hold]);D17=dev[~np.isin(G17[dev],list(overlap17))]\nassert not set(G17[D17])&set(G17[hold])\n\nSHA17=hashlib.sha256((DATA/"train.csv").read_bytes()).hexdigest()\nprint("Development groups",len(D17),flush=True)\n', "import urllib.request,pickle\nGF_MANIFEST27={'repo': 'ctheodoris/Geneformer', 'revision': '1f7fbae4e469a5f4f1af8c111a529cfe1b3829f5', 'model': 'Geneformer-V2-104M', 'embedding_key': 'bert.embeddings.word_embeddings.weight', 'shape': [20275, 768], 'matched': 4377, 'genes': 4384, 'coverage': 0.9984032846715328, 'license': 'apache-2.0', 'files': {'model.safetensors': 'fff5cba29ddd8792991fa77b4872246fbe548a178cebda3775cdc72b67780e7f', 'gene_name_id_dict_gc104M.pkl': 'fabfa0c2f49c598c59ae432a32c3499a5908c033756c663b5e0cddf58deea8e1', 'gene_embedding_table.npz': '30cef571bc6a954bb19d25297d64e61e06dc43e08e07b4cc3a576a10067d43c8', 'config.json': '467d4492f0dd53b4d60afffe20812db484ca1cf9fdbeb6a6e060e93564f70859', 'README.md': '56d6e570b349cbedae9a54634421c94e7af8ea467ce0fdd79193372ae3cbdbd8', 'token_dictionary_gc104M.pkl': '67c445f4385127adfc48dcc072320cd65d6822829bf27dd38070e6e787bc597f'}, 'table_float32_sha256': 'dd8c2ba98e3da00773e82d96c23520e7a760ad7a717d1bf55a8c9a055a90dd75'}\ndef sha_file27(path):\n h=hashlib.sha256()\n with Path(path).open('rb') as f:\n  for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)\n return h.hexdigest()\nclass PrimitiveDictionary27(pickle.Unpickler):\n def find_class(self,module,name):raise pickle.UnpicklingError('Only primitive dictionaries allowed')\ndef load_embedding27():\n folder=OUT/'pretrained';folder.mkdir(parents=True,exist_ok=True);tablepath=folder/'gene_embedding_table.npz'\n if tablepath.exists():\n  with np.load(tablepath,allow_pickle=False) as q:\n   assert q['genes'].tolist()==list(genes);table=q['table'].copy()\n else:\n  mapping={'model.safetensors':'Geneformer-V2-104M/model.safetensors','config.json':'Geneformer-V2-104M/config.json','gene_name_id_dict_gc104M.pkl':'geneformer/gene_name_id_dict_gc104M.pkl','token_dictionary_gc104M.pkl':'geneformer/token_dictionary_gc104M.pkl'}\n  for name,remote in mapping.items():\n   dest=folder/name\n   if not dest.exists():urllib.request.urlretrieve('https://huggingface.co/ctheodoris/Geneformer/resolve/'+GF_MANIFEST27['revision']+'/'+remote,dest)\n   assert sha_file27(dest)==GF_MANIFEST27['files'][name]\n  with (folder/'gene_name_id_dict_gc104M.pkl').open('rb') as f:names=PrimitiveDictionary27(f).load()\n  with (folder/'token_dictionary_gc104M.pkl').open('rb') as f:tokens=PrimitiveDictionary27(f).load()\n  with (folder/'model.safetensors').open('rb') as f:\n   size=int.from_bytes(f.read(8),'little');assert 0<size<10_000_000;meta=json.loads(f.read(size))[GF_MANIFEST27['embedding_key']];assert meta['dtype']=='F32';a,b=meta['data_offsets'];assert 0<=a<b and 8+size+b<= (folder/'model.safetensors').stat().st_size;f.seek(8+size+a);weights=np.frombuffer(f.read(b-a),np.float32).reshape(meta['shape'])\n  table=np.zeros((len(genes),weights.shape[1]),np.float32)\n  for j,g in enumerate(genes):\n   if g in names and names[g] in tokens:table[j]=weights[int(tokens[names[g]])]\n  np.savez_compressed(tablepath,genes=np.array(genes),table=table)\n assert table.shape==(len(genes),768) and np.isfinite(table).all() and hashlib.sha256(table.tobytes()).hexdigest()==GF_MANIFEST27['table_float32_sha256']\n return table\nGF_EMB=load_embedding27();(OUT/'pretrained/manifest.json').write_text(json.dumps(GF_MANIFEST27,indent=2));print('Geneformer fixed gene table',GF_EMB.shape,'coverage',GF_MANIFEST27['coverage'],flush=True)\n", 'KINDS=["missense","silent","nonsense","frameshift","inframe","other"]\nimport scipy.sparse as sp\n\nTRUNC = {"nonsense", "frameshift"}       # 절단형 변이\nNO_HS = {"silent", "other"}              # 핫스팟에서 제외 (기능 변화가 없거나 해석 불가)\n\n# 변이 시그니처 (도메인 지식): 발암 원인마다 남기는 DNA 치환 흔적이 다르다.\n#   자외선(SKCM) → C>T, 흡연(LUAD·LUSC) → C>A, APOBEC(BLCA·CESC) → C>T·C>G\n# 데이터에는 DNA가 없고 아미노산 변화만 있으므로, 표준 유전 암호표(64개 코돈)로\n# "이 아미노산 변화를 만들 수 있는 염기 하나 치환"을 모두 찾아 같은 비중으로 나눠 센다.\n# 치환은 관례대로 피리미딘(C·T) 기준 6종으로 접는다 (G>A는 C>T와 같은 것으로 셈).\nSBS6 = ["C>A", "C>G", "C>T", "T>A", "T>C", "T>G"]\n_B = "TCAG"\nCODON = {a + b + c: aa for (a, b, c), aa in zip(((a, b, c) for a in _B for b in _B for c in _B),\n                                                "FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG")}\n_COMP = {"A": "T", "C": "G", "G": "C", "T": "A"}\n\n\ndef _sig_table():\n    """(바뀌기 전 아미노산, 바뀐 후) → SBS6 비중 벡터(합 1). 염기 하나로 만들 수 없는 변화는 없음."""\n    acc = {}\n    for cod, a in CODON.items():\n        for i in range(3):\n            for x in "ACGT":\n                if x == cod[i]:\n                    continue\n                b = CODON[cod[:i] + x + cod[i + 1:]]\n                r, t = cod[i], x\n                if r in "AG":                       # 퓨린 기준이면 상보 가닥(피리미딘 기준)으로 접기\n                    r, t = _COMP[r], _COMP[t]\n                acc.setdefault((a, b), np.zeros(6))[SBS6.index(f"{r}>{t}")] += 1\n    return {k: v / v.sum() for k, v in acc.items()}\n\n\nSIG = _sig_table()\n\n\nclass FeatureBuilder:\n    def __init__(self, genes, min_gene=5, min_trunc=3, min_hs=3, use_aa=False, min_aa=20, gf_emb=None, use_sig=False):\n        self.genes = genes\n        self.min_gene, self.min_trunc, self.min_hs = min_gene, min_trunc, min_hs\n        self.use_aa, self.min_aa = use_aa, min_aa\n        self.use_sig = use_sig\n        self.gf_emb = gf_emb      # (유전자 수, d) 사전학습 임베딩 표. 고정값이라 fit에서 배울 것이 없음\n\n    @staticmethod\n    def _keys(row):\n        """한 행 → (변이 유전자 집합, 절단형 유전자 집합, 핫스팟 키 집합)"""\n        g = {e[0] for e in row}\n        t = {e[0] for e in row if e[1] in TRUNC}\n        h = {(e[0], f"{e[2]}{e[3]}") for e in row if e[1] not in NO_HS and e[3] >= 0}\n        return g, t, h\n\n    def fit(self, rows, ids):\n        """학습 부분의 행만으로 단어장을 정한다. ids는 누수 검사용으로 기록만 한다."""\n        self.fit_ids = np.asarray(ids)\n        cg, ct, ch, caa = Counter(), Counter(), Counter(), Counter()\n        for row in rows:\n            g, t, h = self._keys(row)\n            cg.update(g); ct.update(t); ch.update(h)\n            if self.use_aa:\n                caa.update((e[2], e[4]) for e in row if e[1] == "missense")\n        self.g_idx = {k: i for i, k in enumerate(sorted(k for k, v in cg.items() if v >= self.min_gene))}\n        self.t_idx = {k: i for i, k in enumerate(sorted(k for k, v in ct.items() if v >= self.min_trunc))}\n        self.h_idx = {k: i for i, k in enumerate(sorted(k for k, v in ch.items() if v >= self.min_hs))}\n        self.aa_idx = {k: i for i, k in enumerate(sorted(k for k, v in caa.items() if v >= self.min_aa))}\n        self.burden_names = ["log_n_genes", "n_events"] + [f"cnt_{k}" for k in KINDS] + [f"frac_{k}" for k in KINDS]\n        G = self.genes\n        self.names = ([f"g_{G[k]}" for k in self.g_idx] + [f"t_{G[k]}" for k in self.t_idx] +\n                      [f"hs_{G[k[0]]}|{k[1]}" for k in self.h_idx] + self.burden_names +\n                      [f"aa_{a}>{b}" for a, b in self.aa_idx] +\n                      ([f"sig_{k}" for k in SBS6] if self.use_sig else []))\n        self.n_lgb = len(self.names)   # LightGBM은 앞의 이 열들만 쓴다 (GF 열은 LR 전용)\n        if self.gf_emb is not None:\n            d = self.gf_emb.shape[1]\n            self.names += [f"gf_mean_{i}" for i in range(d)] + [f"gf_max_{i}" for i in range(d)]\n        return self\n\n    def transform(self, rows):\n        """단어장대로 희소 행렬(float32)로 바꾼다. 각 행은 그 행만 보고 변환된다."""\n        o_t = len(self.g_idx); o_h = o_t + len(self.t_idx); o_b = o_h + len(self.h_idx)\n        o_a = o_b + len(self.burden_names)\n        R, C, V = [], [], []\n        for r, row in enumerate(rows):\n            g, t, h = self._keys(row)\n            cols = ([self.g_idx[k] for k in g if k in self.g_idx] +\n                    [o_t + self.t_idx[k] for k in t if k in self.t_idx] +\n                    [o_h + self.h_idx[k] for k in h if k in self.h_idx])\n            R += [r] * len(cols); C += cols; V += [1.0] * len(cols)\n            cnt = Counter(e[1] for e in row); n_ev = len(row)\n            b = [np.log1p(len(g)), n_ev] + [cnt[k] for k in KINDS] + [cnt[k] / (n_ev + 1) for k in KINDS]\n            R += [r] * len(b); C += list(range(o_b, o_b + len(b))); V += b\n            if self.aa_idx:\n                ca = Counter((e[2], e[4]) for e in row if e[1] == "missense")\n                for k, v in ca.items():\n                    if k in self.aa_idx:\n                        R.append(r); C.append(o_a + self.aa_idx[k]); V.append(v / (cnt["missense"] + 1))\n            if self.use_sig:\n                o_s = o_a + len(self.aa_idx)\n                sv = sum((SIG[(e[2], "*" if e[4] == "X" else e[4])] for e in row\n                          if e[1] in ("missense", "nonsense", "silent") and (e[2], "*" if e[4] == "X" else e[4]) in SIG),\n                         np.zeros(6))\n                if sv.sum() > 0:\n                    sv = sv / sv.sum()\n                R += [r] * 6; C += list(range(o_s, o_s + 6)); V += sv.tolist()\n            if self.gf_emb is not None:\n                gi = sorted({e[0] for e in row if e[1] != "silent"})\n                if gi:          # 변이가 없으면 0으로 둔다\n                    M = self.gf_emb[gi]\n                    b = np.concatenate([M.mean(0), M.max(0)])\n                    o_g = len(self.names) - len(b)\n                    R += [r] * len(b); C += list(range(o_g, o_g + len(b))); V += b.tolist()\n        X = sp.csr_matrix((np.asarray(V, np.float32), (R, C)), shape=(len(rows), len(self.names)))\n        X.eliminate_zeros()\n        return X\nassert CODON["ATG"]=="M" and CODON["TGA"]=="*" and CODON["AAA"]=="K"\nassert all(np.isclose(v.sum(),1) for v in SIG.values())\n', "def rows_for27(frame):\n raw=parse_rows16(canonical_frame25(frame.loc[:,genes]));result=[];gp={g:i for i,g in enumerate(genes)}\n for row in raw:\n  events=[]\n  for g,evs in row['mutations']:\n   for key,kind,pos,parts,flags,dist in evs:\n    typ={'synonymous':'silent','synonymous_block':'silent','stop_gained':'nonsense','deletion':'inframe','insertion':'inframe','duplication':'inframe','delins':'inframe'}.get(kind,kind if kind in {'missense','frameshift'} else 'other')\n    ref=parts[0][1] if parts else (m[0] if (m:=re.match(r'[ACDEFGHIKLMNPQRSTVWY]',key.removeprefix('uncertain|'))) else '')\n    alt=parts[0][2] if len(parts)==1 else ''\n    events.append((gp[g],typ,ref,pos if pos is not None else -1,alt))\n  result.append(events)\n return result\n\ndef new_tree27(rounds):\n return LGBMClassifier(objective='multiclass',class_weight='balanced',num_leaves=15,learning_rate=.05,colsample_bytree=.3,n_estimators=rounds,n_jobs=2,random_state=27027,verbosity=-1)\ndef fit_tree27(indices,sig,seed=27027,rounds=None):\n if rounds is None:\n  aa,bb=next(StratifiedGroupKFold(5,shuffle=True,random_state=seed).split(np.zeros(len(indices)),yi[indices],G17[indices]));it,iv=indices[aa],indices[bb];assert not set(G17[it])&set(G17[iv]);fb=FeatureBuilder(genes,5,3,3,gf_emb=None,use_sig=sig).fit([parsed27[i] for i in it],it);a=fb.transform([parsed27[i] for i in it]);b=fb.transform([parsed27[i] for i in iv]);tree=new_tree27(3000)\n  print('V27 inner tree',sig,len(it),a.shape[1],flush=True);tree.fit(a,yi[it],eval_set=[(b,yi[iv])],callbacks=[early_stopping(100,verbose=False)]);rounds=int(tree.best_iteration_);del tree,fb,a,b;gc.collect()\n fb=FeatureBuilder(genes,5,3,3,gf_emb=None,use_sig=sig).fit([parsed27[i] for i in indices],indices);tree=new_tree27(int(rounds));print('V27 refit tree',sig,len(indices),len(fb.names),rounds,flush=True);tree.fit(fb.transform([parsed27[i] for i in indices]),yi[indices]);return fb,tree,int(rounds)\nclass Model27:\n def __init__(self,sig=True,gf=True):self.sig=sig;self.gf=gf;self.genes_=list(genes);self.classes_=labels.copy()\n def fit(self,indices,tree_bundle=None,rounds=None):\n  self.treepre,self.tree,self.rounds_=tree_bundle if tree_bundle is not None else fit_tree27(indices,self.sig,rounds=rounds)\n  self.lrpre=FeatureBuilder(genes,5,3,3,gf_emb=GF_EMB if self.gf else None,use_sig=self.sig).fit([parsed27[i] for i in indices],indices)\n  self.lr=make_pipeline(MaxAbsScaler(),LogisticRegression(C=.1,class_weight='balanced',max_iter=1000));print('V27 fit LR',self.sig,self.gf,len(indices),len(self.lrpre.names),flush=True);self.lr.fit(self.lrpre.transform([parsed27[i] for i in indices]),yi[indices]);return self\n def predict_parts(self,rows):return self.tree.predict_proba(self.treepre.transform(rows)),self.lr.predict_proba(self.lrpre.transform(rows))\n def predict_rows(self,rows):\n  a,b=self.predict_parts(rows);p=.5*a+.5*b;assert p.shape==(len(rows),len(labels)) and np.isfinite(p).all() and np.allclose(p.sum(1),1);return p\n def predict_frame(self,frame):return self.predict_rows(rows_for27(frame.loc[:,self.genes_]))\n\nparsed27=rows_for27(X)\nassert rows_for27(X.iloc[:12])==rows_for27(legacy_frame25(X.iloc[:12]))\n"]}
HELPER_SOURCE28="import os,sys,json,hashlib,time,subprocess,shutil,tempfile,gc,cloudpickle\nfrom pathlib import Path\nimport numpy as np,pandas as pd\nfrom sklearn.model_selection import StratifiedGroupKFold\nfrom sklearn.metrics import f1_score,accuracy_score\nfrom threadpoolctl import threadpool_limits\nthreadpool_limits(2)\nroots28=[Path.cwd(),*Path.cwd().parents]\nREPO28=next((p for p in roots28 if (p/'data/train.csv').is_file()),None)\nif REPO28 is None:raise FileNotFoundError('data/train.csv is required')\nWORK28=REPO28/'models'/'혜진' if (REPO28/'models/혜진').is_dir() else REPO28\nOUT28=WORK28/'artifacts/v28';OUT28.mkdir(parents=True,exist_ok=True)\ndef guard28(event,args):\n if event=='open' and args and isinstance(args[0],(str,bytes,os.PathLike)) and Path(os.fsdecode(args[0])).name in {'test.csv','sample_submission.csv'}:raise RuntimeError('v28 research cannot open test/sample')\nsys.addaudithook(guard28)\ndef namespace28(version):\n ns={'__name__':'__main__'}\n for i,source in enumerate(COMPONENT_SOURCES28[str(version)]):\n  if version==27 and i==3:\n   dest=ns['OUT']/'pretrained';dest.mkdir(parents=True,exist_ok=True);old=ns['WORK']/'artifacts/v27/pretrained'\n   for name in ['gene_embedding_table.npz','manifest.json']:\n    if (old/name).exists() and not (dest/name).exists():shutil.copy2(old/name,dest/name)\n  exec(compile(source,f'<embedded-v{version}-{i}>','exec'),ns)\n return ns\ndef fresh_model28(ns,version,indices,fold):\n if version==25:return ns['fit25'](indices)\n if version==26:return ns['Model26'](True).fit(indices,seed=28029+fold)\n bundle=ns['fit_tree27'](indices,True,seed=28029+fold)\n return ns['Model27'](True,True).fit(indices,tree_bundle=bundle)\nPLAN28={'v25':.8,'v26':.1,'v27':.1}\nPROTOCOL28={'version':28,'plan':PLAN28,'weight_search':False,'screen_3fold_mean_f1':0.5005193023933087,'confirmation_folds':5,'confirmation_seed':28028,'fold_count_change':'user requested 5-fold after 3-fold screen, before confirmation results','cohort':'same 4960 development patients, old 1241 not evaluated','gate':'mean delta>=0.003, every fold>=-0.001, fixed pooled OOF paired group-bootstrap lower95>0, synthetic noise35 mean delta>=-0.003','noise':'train-fold-only rare pool, new draw seed28029+fold, count35; not claimed to represent real test','all_components_fresh_fit':True,'test_for_research':False,'public_v25_user_reported':0.390417094,'public_target':.50,'limitations':'same repeatedly observed development patients; new split is not independent external patients; v25 legacy-X design followed prior test notation inspection','excluded':['twin label flip','COAD prior adjustment','OOF weight optimization']}\nif not globals().get('IS_WORKER28',False):\n (OUT28/'confirmation_protocol_v28.json').write_text(json.dumps(PROTOCOL28,indent=2));(OUT28/'component_sources.json').write_text(json.dumps(COMPONENT_SOURCES28,ensure_ascii=False))\n"
import os,sys,json,hashlib,time,subprocess,shutil,tempfile,gc,cloudpickle
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score,accuracy_score
from threadpoolctl import threadpool_limits
threadpool_limits(2)
roots28=[Path.cwd(),*Path.cwd().parents]
REPO28=next((p for p in roots28 if (p/'data/train.csv').is_file()),None)
if REPO28 is None:raise FileNotFoundError('data/train.csv is required')
WORK28=REPO28/'models'/'혜진' if (REPO28/'models/혜진').is_dir() else REPO28
OUT28=WORK28/'artifacts/v28';OUT28.mkdir(parents=True,exist_ok=True)
def guard28(event,args):
 if event=='open' and args and isinstance(args[0],(str,bytes,os.PathLike)) and Path(os.fsdecode(args[0])).name in {'test.csv','sample_submission.csv'}:raise RuntimeError('v28 research cannot open test/sample')
sys.addaudithook(guard28)
def namespace28(version):
 ns={'__name__':'__main__'}
 for i,source in enumerate(COMPONENT_SOURCES28[str(version)]):
  if version==27 and i==3:
   dest=ns['OUT']/'pretrained';dest.mkdir(parents=True,exist_ok=True);old=ns['WORK']/'artifacts/v27/pretrained'
   for name in ['gene_embedding_table.npz','manifest.json']:
    if (old/name).exists() and not (dest/name).exists():shutil.copy2(old/name,dest/name)
  exec(compile(source,f'<embedded-v{version}-{i}>','exec'),ns)
 return ns
def fresh_model28(ns,version,indices,fold):
 if version==25:return ns['fit25'](indices)
 if version==26:return ns['Model26'](True).fit(indices,seed=28029+fold)
 bundle=ns['fit_tree27'](indices,True,seed=28029+fold)
 return ns['Model27'](True,True).fit(indices,tree_bundle=bundle)
PLAN28={'v25':.8,'v26':.1,'v27':.1}
PROTOCOL28={'version':28,'plan':PLAN28,'weight_search':False,'screen_3fold_mean_f1':0.5005193023933087,'confirmation_folds':5,'confirmation_seed':28028,'fold_count_change':'user requested 5-fold after 3-fold screen, before confirmation results','cohort':'same 4960 development patients, old 1241 not evaluated','gate':'mean delta>=0.003, every fold>=-0.001, fixed pooled OOF paired group-bootstrap lower95>0, synthetic noise35 mean delta>=-0.003','noise':'train-fold-only rare pool, new draw seed28029+fold, count35; not claimed to represent real test','all_components_fresh_fit':True,'test_for_research':False,'public_v25_user_reported':0.390417094,'public_target':.50,'limitations':'same repeatedly observed development patients; new split is not independent external patients; v25 legacy-X design followed prior test notation inspection','excluded':['twin label flip','COAD prior adjustment','OOF weight optimization']}
if not globals().get('IS_WORKER28',False):
 (OUT28/'confirmation_protocol_v28.json').write_text(json.dumps(PROTOCOL28,indent=2));(OUT28/'component_sources.json').write_text(json.dumps(COMPONENT_SOURCES28,ensure_ascii=False))


In [2]:
WORKER_SOURCE28="import sys,json,gc,time,os\nfrom pathlib import Path\nimport numpy as np,pandas as pd,cloudpickle\nfrom threadpoolctl import threadpool_limits\nthreadpool_limits(2)\nout=Path(sys.argv[1]);version=int(sys.argv[2]);fold=int(sys.argv[3])\nCOMPONENT_SOURCES28=json.loads((out/'component_sources.json').read_text());IS_WORKER28=True\n# Helper source installs train-only guard and contains embedded model definitions.\nexec((out/'helpers.py').read_text())\nns=namespace28(version)\nwith np.load(OUT28/f'confirmation_split{fold}.npz') as q:tr=q['tr'];va=q['va'];dev28=q['dev'];expected_groups=q['group']\nassert np.array_equal(dev28,ns['D17']) and np.array_equal(expected_groups,ns['G17']);assert not set(ns['G17'][tr])&set(ns['G17'][va]);assert not set(ns['G17'][tr])&set(ns['G17'][ns['hold']])\nmodel=fresh_model28(ns,version,tr,fold)\nclean=ns['X'].iloc[va]\nnoise=pd.read_pickle(OUT28/f'synthetic_noise35_fold{fold}.pkl')\nassert list(noise.columns)==list(clean.columns) and len(noise)==len(clean)\np=model.predict_frame(clean);pn=model.predict_frame(noise)\nprobe=clean.iloc[:12];assert np.allclose(model.predict_frame(probe),model.predict_frame(ns['legacy_frame25'](probe)),atol=1e-7)\nassert np.isfinite(p).all() and np.isfinite(pn).all() and np.allclose(p.sum(1),1) and np.allclose(pn.sum(1),1)\ntarget=OUT28/f'confirm_fold{fold}_v{version}.npz';np.savez_compressed(target,tr=tr,va=va,clean=p,noise35=pn,classes=ns['labels'].astype(str))\nmetadata={'version':version,'fold':fold,'fresh_fit':True,'train_sha256':ns['SHA17'],'protocol':PROTOCOL28,'train_rows':len(tr),'validation_rows':len(va),'notation_probe_passed':True}\ntarget.with_suffix('.json').write_text(json.dumps(metadata,indent=2));print('WORKER COMPLETE',version,fold,flush=True)\n"
from IPython.display import display
# Only the baseline namespace is needed to establish grouping and train-fold-only stress inputs.
setup28=namespace28(25);D28=setup28['D17'];G28=setup28['G17'];Y28=setup28['yi'];labels28=setup28['labels'];SHA28=setup28['SHA17'];NC28=len(labels28)
splits28=list(StratifiedGroupKFold(5,shuffle=True,random_state=28028).split(setup28['X'].iloc[D28],Y28[D28],G28[D28]))
for fold,(a,b) in enumerate(splits28):
 tr,va=D28[a],D28[b];assert not set(G28[tr])&set(G28[va]);np.savez_compressed(OUT28/f'confirmation_split{fold}.npz',tr=tr,va=va,dev=D28,group=G28)
 noise_path=OUT28/f'synthetic_noise35_fold{fold}.pkl'
 if not noise_path.exists():setup28['rare_background'](setup28['X'].iloc[va],[setup28['events'][i] for i in tr],28029+fold,count=35).to_pickle(noise_path)
print('5-fold development validation',len(D28),'rows; all models train on about',len(D28)*.8,'rows; old holdout untouched',flush=True)
del setup28;gc.collect()
(OUT28/'worker.py').write_text(WORKER_SOURCE28)
# This file is generated by this notebook, not an external source dependency.
helper_source28=HELPER_SOURCE28
(OUT28/'helpers.py').write_text(helper_source28)
jobs28=[]
for fold in range(5):
 for v in [25,26,27]:
  path=OUT28/f'confirm_fold{fold}_v{v}.npz';meta=path.with_suffix('.json')
  if path.exists() and meta.exists():
   info=json.loads(meta.read_text());assert info['train_sha256']==SHA28 and info['protocol']==PROTOCOL28
   with np.load(path) as q:assert np.array_equal(q['tr'],D28[splits28[fold][0]]) and np.array_equal(q['va'],D28[splits28[fold][1]])
  else:jobs28.append((v,fold))
active28=[]
while jobs28 or active28:
 while jobs28 and len(active28)<2:
  v,fold=jobs28.pop(0);log=OUT28/f'worker_v{v}_fold{fold}.log';handle=log.open('w');env=os.environ.copy();env.update(OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',MKL_NUM_THREADS='2');proc=subprocess.Popen([sys.executable,str((OUT28/'worker.py').resolve()),str(OUT28.resolve()),str(v),str(fold)],stdout=handle,stderr=subprocess.STDOUT,env=env,cwd=str(REPO28));active28.append((proc,handle,v,fold));print('START fresh component',v,'fold',fold,flush=True)
 completed=[]
 for item in active28:
  proc,handle,v,fold=item;code=proc.poll()
  if code is not None:
   handle.close()
   if code:raise RuntimeError(f'v{v} fold{fold} failed; inspect '+str(OUT28/f'worker_v{v}_fold{fold}.log'))
   print('FINISH fresh component',v,'fold',fold,flush=True);completed.append(item)
 active28=[item for item in active28 if item not in completed]
 if active28:time.sleep(2)
confirmation28=[];oof28={k:np.zeros((len(D28),NC28)) for k in ['baseline','candidate','baseline_noise35','candidate_noise35']}
f28=lambda y,p:float(f1_score(y,p.argmax(1),labels=np.arange(NC28),average='macro',zero_division=0))
for fold,(a,b) in enumerate(splits28):
 va=D28[b];pred={};noisy={}
 for v in [25,26,27]:
  with np.load(OUT28/f'confirm_fold{fold}_v{v}.npz') as q:assert np.array_equal(q['va'],va) and np.array_equal(q['classes'],labels28);pred[f'v{v}']=q['clean'];noisy[f'v{v}']=q['noise35']
 base=pred['v25'];candidate=sum(w*pred[k] for k,w in PLAN28.items());bn=noisy['v25'];cn=sum(w*noisy[k] for k,w in PLAN28.items())
 oof28['baseline'][b]=base;oof28['candidate'][b]=candidate;oof28['baseline_noise35'][b]=bn;oof28['candidate_noise35'][b]=cn
 confirmation28.append({'fold':fold,'baseline_f1':f28(Y28[va],base),'candidate_f1':f28(Y28[va],candidate),'delta':f28(Y28[va],candidate)-f28(Y28[va],base),'baseline_accuracy':float(accuracy_score(Y28[va],base.argmax(1))),'candidate_accuracy':float(accuracy_score(Y28[va],candidate.argmax(1))),'noise35_delta':f28(Y28[va],cn)-f28(Y28[va],bn)})
table28=pd.DataFrame(confirmation28);table28.to_csv(OUT28/'confirmation_results.csv',index=False);np.savez_compressed(OUT28/'confirmation_oof.npz',dev=D28,classes=labels28.astype(str),**oof28)
rng=np.random.default_rng(28030);unique=np.unique(G28[D28]);positions={g:np.flatnonzero(G28[D28]==g) for g in unique};bootstrap=[]
for _ in range(500):
 ids=np.concatenate([positions[g] for g in rng.choice(unique,size=len(unique),replace=True)]);bootstrap.append(f28(Y28[D28[ids]],oof28['candidate'][ids])-f28(Y28[D28[ids]],oof28['baseline'][ids]))
ci=np.quantile(bootstrap,[.025,.975]).tolist();gate=bool(table28.delta.mean()>=.003 and table28.delta.min()>=-.001 and ci[0]>0 and table28.noise35_delta.mean()>=-.003)
verification28={'version':28,'confirmation_folds':5,'baseline_mean_f1':float(table28.baseline_f1.mean()),'candidate_mean_f1':float(table28.candidate_f1.mean()),'candidate_mean_accuracy':float(table28.candidate_accuracy.mean()),'mean_delta':float(table28.delta.mean()),'group_bootstrap_pooled_delta_ci95':ci,'noise35_mean_delta':float(table28.noise35_delta.mean()),'confirmation_passed':gate,'confirmation_performed':True,'all_components_fresh_fit':True,'public_v25':.390417094,'public_v28':None,'model_saved':False,'test_for_research':False,'train_sha256':SHA28,'limitations':'new partition of repeatedly observed development patients; bootstrap conditional on fixed OOF predictions, not whole fitting/selection uncertainty'}
(OUT28/'verification_v28.json').write_text(json.dumps(verification28,indent=2));print(table28.to_string(index=False),flush=True);print('V28 CONFIRMATION',json.dumps(verification28,indent=2),flush=True)


TRAIN ONLY (6201, 4386) Dev/holdout 4960 1241 OUT /Users/mac/project/cancer_project/artifacts/v28/definitions_v25
Parsed rows: 6201 Normalization examples passed
Novel settings: 44 Views: {'raw': (1.0, 1.0), 'syn_down': (1.0, 0.25), 'gated': (0.25, 1.0), 'gated_syn_down': (0.25, 0.25), 'gated_nonsyn': (0.25, 0.0)}
Total new configurations after refinement: 59
New configurations 8
Total new configurations including augmentation extension: 12
V10 candidates: 19 semantic checks passed
Extended candidates: 24
P43 annotation safety fixtures passed: 15
Literature panel subsets: 15 (external biological prior; no external patient data)
5-fold development validation 4960 rows; all models train on about 3968.0 rows; old holdout untouched
 fold  baseline_f1  candidate_f1    delta  baseline_accuracy  candidate_accuracy  noise35_delta
    0     0.519470      0.519863 0.000393           0.498992            0.500000      -0.021297
    1     0.487884      0.495989 0.008104           0.483384          

## 실행 중 집계 형식 복구 기록
최초 실행에서 15개 구성 모델 학습은 모두 완료됐으나, class 이름을 저장한 NumPy object 배열을 기본 로더로 읽을 때 집계가 중단됐습니다. class 메타데이터만 Unicode 문자열로 바꾸고 모든 split 인덱스와 예측 확률이 byte 단위로 동일함을 확인했습니다. 원본 캐시는 recovery_original_object_labels/, 내역은 dtype_recovery.json, 최초 학습 로그는 first_attempt_training.log에 보존했습니다. 최종 집계는 이 검증된 새 학습 결과를 재사용했으며 추가 fitting이나 비중 변경은 하지 않았습니다. 노트북의 새 worker는 처음부터 Unicode로 저장합니다.

## train-only 형제 라벨 구조 별도 감사
아래는 데이터 구조 진단일 뿐 기본 모델에 반전 규칙을 적용하지 않습니다. 무작위 행 분할의 조건부 정확도를 그룹 일반화 성능에 섞지 않습니다.

In [3]:
from pathlib import Path
import os,sys,json,hashlib,collections
import numpy as np,pandas as pd
from sklearn.model_selection import StratifiedKFold
R=WORK28;out=OUT28
def guard(e,a):
 if e=='open' and a and isinstance(a[0],(str,bytes,os.PathLike)) and Path(os.fsdecode(a[0])).name in {'test.csv','sample_submission.csv'}:raise RuntimeError('label audit is train-only')
sys.addaudithook(guard)
t=pd.read_csv(REPO28/'data/train.csv').fillna('WT');genes=[c for c in t if c not in ['ID','SUBCLASS']];h=np.array([hashlib.sha256('|'.join(row).encode()).hexdigest() for row in t[genes].to_numpy(dtype=object)]);y=t.SUBCLASS.to_numpy();groups=collections.defaultdict(list)
for i,key in enumerate(h):groups[key].append(i)
swap={'KIRC':'KIPAN','KIPAN':'KIRC','LGG':'GBMLGG','GBMLGG':'LGG'}
pair_counts=collections.Counter();duplicates=0;multi_label=0
for ix in groups.values():
 if len(ix)>1:
  duplicates+=1;s=tuple(sorted(set(y[ix])));pair_counts[' / '.join(s)]+=1;multi_label+=len(s)>1
def lookup(tr,va):
 known=collections.defaultdict(set)
 for i in tr:known[h[i]].add(y[i])
 matched=[];correct=[]
 for i in va:
  seen=known.get(h[i],set())
  if len(seen)==1 and (label:=next(iter(seen))) in swap:matched.append(i);correct.append(swap[label]==y[i])
 return len(matched),sum(correct)
random=[]
for k,(tr,va) in enumerate(StratifiedKFold(5,shuffle=True,random_state=42).split(np.zeros(len(t)),y)):
 n,c=lookup(tr,va);random.append({'fold':k,'matched_rows':n,'correct_under_flip':int(c),'conditional_accuracy':c/n if n else None})
grouped=[]
for k in range(5):
 with np.load(out/f'confirmation_split{k}.npz') as q:n,c=lookup(q['tr'],q['va']);grouped.append({'fold':k,'matched_rows':n})
proof={'train_only':True,'raw_identical_profile_groups':duplicates,'multi_label_duplicate_groups':multi_label,'duplicate_group_label_sets':dict(pair_counts),'random_row_5fold_diagnostic':random,'random_matched_total':sum(r['matched_rows'] for r in random),'random_conditional_accuracy':sum(r['correct_under_flip'] for r in random)/sum(r['matched_rows'] for r in random),'group_5fold_matches':grouped,'test_examined':False,'rule_applied_to_submission':False,'limitations':'identical observed mutation profiles do not establish patient identity; random-row diagnostic measures duplicate-structure exploitation and cannot establish independent patient generalization; opposite label on competition test not guaranteed; official competition rule page could not be retrieved due anti-bot challenge, no approval inferred'}
(out/'label_structure_audit.json').write_text(json.dumps(proof,indent=2));print(json.dumps(proof,indent=2))
text='# v28 형제 라벨 구조 별도 감사 — 기본 모델에 미적용\n\n실제 train만 읽어 확인했습니다. test는 열지 않았고 정답 반전 규칙을 제출에 넣지 않았습니다.\n\n```json\n'+json.dumps(proof,ensure_ascii=False,indent=2)+'\n```\n\n같은 변이 프로필과 환자 동일성은 구분해야 합니다. 무작위 행 분할에서 동일 프로필이 양쪽에 남을 때의 조건부 정확도는 중복 구조를 이용한 진단이며, 새로운 환자에 대한 성능이 아닙니다. 그룹 분할에서는 이 규칙의 효과를 측정할 수 없습니다. test의 반대 라벨이 항상 정답이라는 보장도 없으며, 이번 v28 기본 예측에는 적용하지 않았습니다.\n\n[해당 대회 규칙 페이지](https://dacon.io/competitions/official/236355/overview/rules)는 자동 접근 시 JavaScript/로봇 확인으로 본문을 읽지 못했습니다. 다른 대회의 일반 규칙으로 허용 또는 금지를 대신 판정하지 않았습니다.\n'
(R/'LABEL_STRUCTURE_AUDIT_V28.md').write_text(text)


{
  "train_only": true,
  "raw_identical_profile_groups": 451,
  "multi_label_duplicate_groups": 447,
  "duplicate_group_label_sets": {
    "KIPAN / KIRC": 272,
    "GBMLGG / LGG": 173,
    "BRCA / CESC / GBMLGG / KIPAN / KIRC / LAML / LGG / LUSC / PAAD / PCPG / PRAD / SARC / STES / THCA / THYM": 1,
    "LGG / THCA": 1,
    "LAML": 3,
    "KIPAN": 1
  },
  "random_row_5fold_diagnostic": [
    {
      "fold": 0,
      "matched_rows": 147,
      "correct_under_flip": 146,
      "conditional_accuracy": 0.9931972789115646
    },
    {
      "fold": 1,
      "matched_rows": 145,
      "correct_under_flip": 145,
      "conditional_accuracy": 1.0
    },
    {
      "fold": 2,
      "matched_rows": 143,
      "correct_under_flip": 143,
      "conditional_accuracy": 1.0
    },
    {
      "fold": 3,
      "matched_rows": 148,
      "correct_under_flip": 147,
      "conditional_accuracy": 0.9932432432432432
    },
    {
      "fold": 4,
      "matched_rows": 134,
      "correct_under_flip": 134,

## 전체 train 구성 모델의 고정 배포 조합 저장
배포에서만 기존 전체 train 모델을 재사용합니다. 5-fold 확인은 해당 fold의 새 학습 모델만 사용했습니다. 이전 전체 모델 파일이 없으면 포함된 원래 학습 레시피로 재학습합니다. 확인 기준 미달이면 실험용으로만 보존하고 새 최고 모델로 추천하지 않습니다.

In [4]:
class FixedBlend28:
 def __init__(self,models):
  self.models=models;self.weights_=dict(PLAN28);self.genes_=list(models['v25'].genes_);self.classes_=np.asarray(models['v25'].classes_).copy();self.training_data_sha256_=SHA28;self.training_protocol_=dict(PROTOCOL28)
  for model in models.values():assert list(model.genes_)==self.genes_ and np.array_equal(model.classes_,self.classes_)
 def predict_frame(self,frame):
  frame=frame.loc[:,self.genes_];p=sum(self.weights_[name]*model.predict_frame(frame) for name,model in self.models.items());assert p.shape==(len(frame),len(self.classes_)) and np.isfinite(p).all() and (p>=0).all() and np.allclose(p.sum(1),1);return p
full_paths28={'v25':('artifacts/v25/cancer_model_v25_legacy_stop_retrained.pkl','cc7cbf0b30c5934212aa2ef6a5ec64ec1f045be323dbda5a949fe4304ec0b9d9'),'v26':('artifacts/v26/cancer_model_v26_minjung_features_experimental.pkl','20e7af04f05834490657b817195aeffe892b3c1e29d58a5e5f046866ca5d63fc'),'v27':('artifacts/v27/cancer_model_v27_geneformer_proxy_experimental.pkl','053d1cfb8ea667a50cc0d43c817e2a2aa1a8279ad8bb098eaa3cb74f9124d604')}
models28={};full_provenance28={}
for key,(relative,expected) in full_paths28.items():
 path=WORK28/relative
 if path.exists():
  assert hashlib.sha256(path.read_bytes()).hexdigest()==expected,'Existing component differs from verified source; preserve and review.'
  with path.open('rb') as f:models28[key]=cloudpickle.load(f)
  assert models28[key].training_data_sha256_==SHA28
  full_provenance28[key]={'mode':'verified existing full-train model reused for deployment only','sha256':expected}
 else:
  v=int(key[1:]);ns=namespace28(v);indices=np.arange(len(ns['train']))
  if v==25:models28[key]=ns['fit25'](indices)
  elif v==26:models28[key]=ns['Model26'](True).fit(indices,rounds=165)
  else:models28[key]=ns['Model27'](True,True).fit(indices,rounds=98)
  models28[key].training_data_sha256_=SHA28;full_provenance28[key]={'mode':'cache absent: recreated from embedded original full-fit recipes; tree rounds v26=165, v27=98, not reselected from v28 outcomes'}
  del ns;gc.collect()
final28=FixedBlend28(models28);final28.component_provenance_=full_provenance28
path28=OUT28/'cancer_model_v28_fixed_blend_experimental.pkl'
with path28.open('wb') as f:cloudpickle.dump(final28,f)
probe28=pd.read_csv(REPO28/'data/train.csv',nrows=12).loc[:,final28.genes_];gold28=final28.predict_frame(probe28)
assert np.allclose(final28.predict_frame(probe28.iloc[[7,2,0]]),gold28[[7,2,0]],atol=1e-6)
with path28.open('rb') as f:reloaded28=cloudpickle.load(f)
assert np.allclose(gold28,reloaded28.predict_frame(probe28),atol=1e-7)
verification28.update(model_saved=True,model_sha256=hashlib.sha256(path28.read_bytes()).hexdigest(),model_bytes=path28.stat().st_size,final_training='fixed ensemble of separately verified all-train components; no test fit; confirmation used fresh fold fits only',component_provenance=full_provenance28)
(OUT28/'verification_v28.json').write_text(json.dumps(verification28,indent=2));print('V28 MODEL SAVED',json.dumps(verification28,indent=2),flush=True)


V28 MODEL SAVED {
  "version": 28,
  "confirmation_folds": 5,
  "baseline_mean_f1": 0.49126981877812226,
  "candidate_mean_f1": 0.4995781526623018,
  "candidate_mean_accuracy": 0.49516454194202686,
  "mean_delta": 0.008308333884179486,
  "group_bootstrap_pooled_delta_ci95": [
    0.0029558323412349295,
    0.014218870635395937
  ],
  "noise35_mean_delta": -0.014166312109507861,
  "confirmation_passed": false,
  "confirmation_performed": true,
  "all_components_fresh_fit": true,
  "public_v25": 0.390417094,
  "public_v28": null,
  "model_saved": true,
  "test_for_research": false,
  "train_sha256": "92418b8441d058cfc68e939dd88725610750be4bc8edc51253cffc72fc4fc0ab",
  "limitations": "new partition of repeatedly observed development patients; bootstrap conditional on fixed OOF predictions, not whole fitting/selection uncertainty",
  "clean_mean_gate_passed": true,
  "fold_safety_gate_passed": true,
  "bootstrap_gate_passed": true,
  "noise35_gate_passed": false,
  "component_sources_sha25

## 고정 최종 모델로 test 추론
연구 프로세스의 접근 차단은 유지하고 별도 자식 프로세스에서 저장 모델 predict만 실행합니다. 5-fold 확인 기준 통과 여부와 Public 미확인을 구분합니다.

In [5]:
# 별도 프로세스에서 고정 모델로 최종 test 예측만 수행합니다.
from pathlib import Path
import subprocess,sys
roots28=[Path.cwd(),*Path.cwd().parents]
repo28=next((p for p in roots28 if (p/'data/train.csv').is_file()),None)
if repo28 is None:raise FileNotFoundError('data 폴더가 필요합니다.')
work28=repo28/'models'/'혜진' if (repo28/'models'/'혜진').is_dir() else repo28
program28="import sys,os,json,hashlib,cloudpickle,numpy as np,pandas as pd\nfrom pathlib import Path\nfrom threadpoolctl import threadpool_limits\nthreadpool_limits(2)\ndef guard(e,a):\n if e=='open' and a and isinstance(a[0],(str,bytes,os.PathLike)) and Path(os.fsdecode(a[0])).name=='train.csv':raise RuntimeError('Final inference does not read train.csv')\nsys.addaudithook(guard)\nmp,tp,sp,op=map(Path,sys.argv[1:5]);proof=json.loads((mp.parent/'verification_v28.json').read_text());assert proof['model_saved']\nassert hashlib.sha256(mp.read_bytes()).hexdigest()==proof['model_sha256']\nwith mp.open('rb') as f:model=cloudpickle.load(f)\nt=pd.read_csv(tp);s=pd.read_csv(sp);assert t.ID.is_unique and s.ID.is_unique and len(t)==len(s) and set(t.ID)==set(s.ID)\nx=t.loc[:,model.genes_];p=model.predict_frame(x);assert p.shape==(len(t),26) and np.isfinite(p).all() and (p>=0).all() and np.allclose(p.sum(1),1)\nidx=[10,3,500,0];assert np.allclose(model.predict_frame(x.iloc[idx]),p[idx],atol=1e-6)\ns.SUBCLASS=s.ID.map(pd.Series(model.classes_[p.argmax(1)],index=t.ID));assert not s.isna().any().any()\nop.parent.mkdir(parents=True,exist_ok=True)\npayload=s.to_csv(index=False).encode()\nif op.exists():assert op.read_bytes()==payload,'Existing v28 submission differs; preserve it.'\nelse:op.write_bytes(payload)\nassert pd.read_csv(op).ID.equals(s.ID)\nreport={'version':28,'rows':len(s),'model_sha256':proof['model_sha256'],'csv_sha256':hashlib.sha256(op.read_bytes()).hexdigest(),'phase':'final inference only','model_fitting':False,'test_used_for_validation':False,'public_score':None,'batch_invariance':True}\n(mp.parent/'submission_verification.json').write_text(json.dumps(report,ensure_ascii=False,indent=2));print(json.dumps(report,ensure_ascii=False,indent=2))\n"
args28=[str(work28/'artifacts/v28/cancer_model_v28_fixed_blend_experimental.pkl'),str(repo28/'data/test.csv'),str(repo28/'data/sample_submission.csv'),str(repo28/'submission/혜진_submission_v28_fixed_blend_experimental.csv')]
run28=subprocess.run([sys.executable,'-c',program28,*args28],capture_output=True,text=True)
if run28.returncode:raise RuntimeError(run28.stderr)
print(run28.stdout)


{
  "version": 28,
  "rows": 2546,
  "model_sha256": "69fad15754a57b9fb90a2266c7c0f361dbeeee3398a66d0daf011318bb3ac001",
  "csv_sha256": "8f06574a8ede8b1ed2e5e3bf291063224fc6371d9e2865882f459de5d3a448ce",
  "phase": "final inference only",
  "model_fitting": false,
  "test_used_for_validation": false,
  "public_score": null,
  "batch_invariance": true
}